# Ensemble: combining the regression models

Implements [`.claude/specs/08-ensemble.md`](../../.claude/specs/08-ensemble.md).

Spec 06's models win in different situations: the direct boosters on high residual load, the
hybrids and the random forest in the low tail, the linear model on ordinary hours. This notebook
**combines their forecasts** and checks whether the combination beats the **best single model**,
not only SMARD, on identical hours.

**Nothing is refitted here.** The members are read from spec 06's committed model saves in
`data/models/<model_key>/`, which hold every run's forecasts, the validation year included. The
notebook fits no base model; a full run takes about 4 minutes, most of it the out-of-fold refits
of the combination methods in §4.

## How to use this notebook

- Change a value in the configuration ([§1.3](#sec-1-3)), then re-run top to bottom: which members
  take part (`MEMBERS`), which combination methods run (`METHODS`), the regime split (`REGIMES`,
  `SHRINK_HOURS`) and the windows of the adaptive method (`ADAPTIVE_WINDOWS`).
- Every weight, bin edge and window is chosen on the **validation year only**. The test year is
  scored once, like every model in spec 06.
- The interpretation is written **once**, in the closing section, for the default configuration.

## Conventions

- **Sign:** `error = forecast − actual`, as in spec 04. **Positive = over-forecast.**
- **Units:** forecasts and errors in `MWh`, weights without unit, skill and coverage in `%`.
- **Durations, never row counts.** No literal calendar year or date appears in code.
- `time_series` holds exactly `SERIES + DERIVED`. Everything this notebook builds lives in separate
  frames.


---

## 1 Setup and inputs

### 1.1 Shared setup

The same setup as [`team-EDA.ipynb`](../01_eda/team-EDA.ipynb) §1, as reused by
[`regression-models-claude.ipynb`](regression-models-claude.ipynb) §1.1: it finds `data/`, loads
`data/smard.csv` into `time_series` and builds `SERIES`, `DERIVED`, `YEARS` and
`style_timeseries`. Nothing is re-derived here.


In [ ]:
# Standard library
import hashlib
import json
import time
import warnings
from pathlib import Path

# Data handling and plotting
import joblib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.optimize import linprog
from sklearn.linear_model import Ridge

# Find the `data/` folder: first in the working directory, then in each parent directory.
DATA_DIR = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "data").is_dir():
        DATA_DIR = folder / "data"
        break

if DATA_DIR is None:
    raise RuntimeError(
        f"No data/ folder found in {Path.cwd()} or any parent folder. "
        "Run the notebook from inside the repository."
    )

# data/ is gitignored, so a fresh clone has no smard.csv yet.
DATA = DATA_DIR / "smard.csv"
if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone. "
        "Create it with a full run of notebooks/API-connection.ipynb."
    )

print(
    f"pandas {pd.__version__} · numpy {np.__version__} · scipy {scipy.__version__} · "
    f"scikit-learn {sklearn.__version__} · joblib {joblib.__version__}"
)
print(f"Data file: {DATA}")

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Apply the EDA time-series chart style to the matplotlib axes.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, color="grey")

    # Light horizontal grid lines behind the data, no box on the top and right
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    # Tick labels without tick marks
    ax.tick_params(colors="black", length=0)

    # x axis: one label per year, minor ticks every quarter
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # y axis: thousands separator, no decimals (e.g. 45,000)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")

In [ ]:
# Column names in smard.csv (see API-connection.ipynb)
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
    "Capacity Wind Offshore": "cap_wind_off",
    "Capacity Wind Onshore": "cap_wind_on",
    "Capacity Solar": "cap_solar",
}

# The CSV uses the German Excel format: ";" between columns and "," as the decimal mark.
raw = pd.read_csv(DATA, sep=";", decimal=",", encoding="utf-8-sig")
assert set(raw.columns) == {"timestamp"} | COLUMNS.keys(), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | COLUMNS.keys()))}"
)

raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")
time_series = raw.set_index("timestamp").sort_index()
del raw  # only time_series is used from here on

assert all(
    pd.api.types.is_float_dtype(time_series[column]) for column in COLUMNS.values()
), time_series.dtypes

# Snapshot of the loaded data, for the closing self-check
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"time_series: {len(time_series):,} rows x {time_series.shape[1]} columns")
print(f"time span  : {LOADED['start']}  ->  {LOADED['end']}")

In [ ]:
# The data columns loaded from smard.csv, in the same order as COLUMNS
SERIES = list(COLUMNS.values())

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons. December counts towards the FOLLOWING year's winter
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

# Columns derived from the data and the timestamps
time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)
time_series["spans_gap"] = time_series.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow",
    "is_weekend", "date", "season", "season_year", "spans_gap",
]
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

YEARS = sorted(int(year) for year in time_series["year"].unique())

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS = {YEARS}")

### 1.2 Resolution and durations

The time step is measured from the data. Every window below is a duration and is converted to a
number of rows only through `n_obs`, as in spec 06 §1.2.


In [ ]:
# Time step between two rows, measured from the data (the most common gap)
RESOLUTION = time_series.index.to_series().diff().mode().iloc[0]

# A day is complete with at least 23 of 24 hours: accepts the spring-DST day
DAY_COMPLETENESS = 23 / 24


def n_obs(duration):
    """Number of rows `duration` spans at RESOLUTION; stops if it is not a whole number."""
    count = duration / RESOLUTION
    if count != int(count):
        raise ValueError(f"{duration} is not a whole multiple of the resolution {RESOLUTION}")
    return int(count)


def describe(value):
    """Short readable text for a duration, e.g. '24h' or '7 days'."""
    if isinstance(value, pd.Timedelta):
        if value >= pd.Timedelta(days=2) and value == pd.Timedelta(days=value.days):
            return f"{value.days} days"
        return f"{value / pd.Timedelta(hours=1):g}h"
    return str(value)


EXPECTED_OBS_PER_DAY = n_obs(pd.Timedelta(days=1))
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))
print(f"resolution {describe(RESOLUTION)}: {EXPECTED_OBS_PER_DAY} observations per full day")

### <a id="sec-1-3"></a>1.3 Configuration

The values to play with. Change one, then re-run the notebook top to bottom.

- **`MEMBERS`**: the forecasts the ensembles combine. `None` takes every model save found in
  `data/models/` ([§1.5](#sec-1-5)). A list picks members by key, e.g.
  `["lgbm_direct", "linear_direct", "random_forest_hybrid"]`.
- **`USE_SMARD_MEMBER`** / **`USE_NAIVE_MEMBER`**: add SMARD's forecast or seasonal naive as a
  member. SMARD is on by default (spec 08, see its Open questions); switching it off only changes
  the members, SMARD stays in every scoreboard as the benchmark.
- **`METHODS`**: the combination methods (§3), each with an `enabled` switch.
- **`REGIMES`**, **`SHRINK_HOURS`**: how `regime_weighted` splits the hours and how strongly a small
  regime is pulled toward the global weights.
- **`ADAPTIVE_WINDOWS`**: candidate trailing windows for `adaptive`; the validation year picks one.


In [ ]:
SEED = 42  # every random draw in this notebook

# Members: None = every model save in data/models/, or a list of model keys
MEMBERS = None
USE_SMARD_MEMBER = True    # SMARD's fc_residual_load as a member (always in the scoreboards)
USE_NAIVE_MEMBER = False   # seasonal naive DAY−7 as a member (always in the scoreboards)

# Combination methods (§3). The keys become "ensemble_<key>" in the exports.
METHODS = {
    "equal_mean": {
        "enabled": True,
        "label": "Equal mean",
        "color": "#1F77B4",
    },
    "weighted": {
        "enabled": True,
        "label": "Weighted (validation MAE)",
        "color": "#D62728",
    },
    "regime_weighted": {
        "enabled": True,
        "label": "Regime-weighted",
        "color": "#2CA02C",
    },
    "adaptive": {
        "enabled": True,
        "label": "Adaptive (trailing MAE)",
        "color": "#9467BD",
    },
    "stacking": {
        "enabled": False,      # off by default: the most parameters, overlaps regime_weighted
        "alpha": 10.0,         # Ridge penalty of the meta-model
        "label": "Stacking (Ridge)",
        "color": "#8C564B",
    },
}

# Regimes of regime_weighted (§3): by the members' mean forecast, known at issue time
REGIMES = {
    "level": True,     # below_zero / low / ordinary / high / high_extreme
    "season": False,   # also split by meteorological season
}
LEVEL_ZERO = 0.0                      # MWh: fixed edge, the zero basis of risk-definition.ipynb
LEVEL_QUANTILES = [0.25, 0.75, 0.99]  # further edges: quantiles of the validation year
SHRINK_HOURS = 100                    # k in n / (n + k): pull of a small regime toward the global weights

# adaptive (§3): candidate trailing windows, the validation year picks one
ADAPTIVE_WINDOWS = [pd.Timedelta(days=7), pd.Timedelta(days=30)]

INTERVAL = {"level": 0.95}     # prediction interval (§4)
PLOT_METHOD = None             # None: the enabled ensemble with the lowest test MAE
PLOT_SPLIT_METHOD = "rolling"  # falls back to "static" if the saves have no rolling runs
EXPORT_ENABLED = True          # True writes data/models/ensemble_*.csv (§6)

print(f"MEMBERS          : {MEMBERS if MEMBERS else 'every model save'}")
print(f"SMARD as member  : {USE_SMARD_MEMBER} · seasonal naive as member: {USE_NAIVE_MEMBER}")
print(f"METHODS enabled  : {[key for key, method in METHODS.items() if method['enabled']]}")
print(f"REGIMES          : {[key for key, on in REGIMES.items() if on]}, "
      f"edges {LEVEL_ZERO:g} MWh + quantiles {LEVEL_QUANTILES}, SHRINK_HOURS = {SHRINK_HOURS}")
print(f"ADAPTIVE_WINDOWS : {[describe(window) for window in ADAPTIVE_WINDOWS]}")
print(f"INTERVAL         : {INTERVAL['level']:.0%} · PLOT: {PLOT_METHOD or 'best ensemble'}, "
      f"{PLOT_SPLIT_METHOD} · EXPORT_ENABLED: {EXPORT_ENABLED}")

### 1.4 SMARD's hourly errors

SMARD's forecast comes from spec 04's export, the source of truth for re-scoring SMARD on any
hours. It is loaded the same way as in spec 06 §1.4 and must match `time_series` exactly.


In [ ]:
SMARD_ERRORS = DATA_DIR / "metrics" / "smard_forecast_errors_hourly.csv"
SMARD_ERRORS_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SMARD_ERROR_COLUMNS = [
    "residual_load",
    "fc_residual_load",
    "err_residual_load",
    "err_grid_load",
    "err_renewables",
]

if not SMARD_ERRORS.exists():
    raise FileNotFoundError(
        f"{SMARD_ERRORS} not found. Create it by running {SMARD_ERRORS_SOURCE} top to bottom."
    )

smard_errors = pd.read_csv(SMARD_ERRORS)
smard_errors["timestamp"] = pd.to_datetime(smard_errors["timestamp"], format="%Y-%m-%d %H:%M:%S")
smard_errors = smard_errors.set_index("timestamp")

missing_columns = sorted(set(SMARD_ERROR_COLUMNS) - set(smard_errors.columns))
if missing_columns:
    raise ValueError(f"{SMARD_ERRORS.name} lacks {missing_columns}. Re-run {SMARD_ERRORS_SOURCE}.")

if not smard_errors.index.equals(time_series.index):
    raise ValueError(
        f"{SMARD_ERRORS.name} does not match data/smard.csv "
        f"({len(smard_errors):,} against {len(time_series):,} rows). "
        f"Re-run {SMARD_ERRORS_SOURCE} top to bottom."
    )

print(f"smard_errors: {len(smard_errors):,} rows, index identical to time_series")

### <a id="sec-1-5"></a>1.5 The members: spec 06's model saves

Every folder in `data/models/` with a `config.json` and a `results.joblib` is one saved model
(spec 06 Behaviour 34). `results.joblib` holds `RESULTS[model_key]`; this notebook takes the
forecast series of its four runs from `runs[run][0]`:

| run | used for |
|---|---|
| `validation_rolling`, `validation_static` | learning weights and calibrating the bands |
| `test_rolling`, `test_static` | scoring, once |

**A save's forecasts are only valid on the data they were run on.** The month hashes in its
`config.json` (`run_on`) are compared with the data loaded here, using the same `file_snapshot`
helper as spec 06 §5, copied. All saves must also share one forecast setting (`DATA_INFO`), one
set of windows and one set of split methods; members from different settings cannot be combined.
The device (CPU / GPU) and the library versions are printed, never compared.

`results.joblib` is a pickle and can run code when loaded: load only the team's committed saves.


In [ ]:
MODEL_STORE = DATA_DIR / "models"
SAVE_FILES = {"config": "config.json", "results": "results.joblib"}
RUN_NAMES = ["validation_rolling", "validation_static", "test_rolling", "test_static"]
INPUT_FILES = {
    "smard.csv": time_series[SERIES],
    SMARD_ERRORS.name: smard_errors[SMARD_ERROR_COLUMNS],
}


def file_snapshot(frame):
    """Rows, span and one hash per calendar month of an input file's values (spec 06 §5)."""
    row_hashes = pd.util.hash_pandas_object(frame, index=True)
    month_hashes = {}
    for month, hashes in row_hashes.groupby(frame.index.strftime("%Y-%m")):
        month_hashes[month] = hashlib.sha256(hashes.to_numpy().tobytes()).hexdigest()[:16]
    return {
        "rows": len(frame),
        "start": str(frame.index[0]),
        "end": str(frame.index[-1]),
        "month_hashes": month_hashes,
    }


CURRENT_DATA = {name: file_snapshot(frame) for name, frame in INPUT_FILES.items()}


def data_differences(saved):
    """One line per input file that differs from the save: its span, or the months."""
    lines = []
    for name in INPUT_FILES:
        old, new = saved[name], CURRENT_DATA[name]
        old_span = f"{old['rows']:,} rows, {old['start']} .. {old['end']}"
        new_span = f"{new['rows']:,} rows, {new['start']} .. {new['end']}"
        if old_span != new_span:
            lines.append(f"{name}: saved {old_span}, now {new_span}")
        shared = sorted(old["month_hashes"].keys() & new["month_hashes"].keys())
        changed = [m for m in shared if old["month_hashes"][m] != new["month_hashes"][m]]
        if changed:
            lines.append(f"{name}: values differ in {', '.join(changed)}")
    return lines


# Every complete save in data/models/
SAVES, SKIPPED = {}, {}
for folder in sorted(path for path in MODEL_STORE.iterdir() if path.is_dir()):
    paths = {name: folder / file for name, file in SAVE_FILES.items()}
    if not all(path.exists() for path in paths.values()):
        SKIPPED[folder.name] = "no config.json + results.joblib"
        continue
    config = json.loads(paths["config"].read_text(encoding="utf-8"))
    result = joblib.load(paths["results"])
    missing_runs = [run for run in RUN_NAMES if run not in result.get("runs", {})]
    if missing_runs:
        SKIPPED[folder.name] = f"runs missing: {missing_runs}"
        continue
    SAVES[folder.name] = {
        "config": config,
        "runs": {run: result["runs"][run][0] for run in RUN_NAMES},
    }

if not SAVES:
    raise FileNotFoundError(
        f"No complete model save in {MODEL_STORE}. Run regression-models-claude.ipynb "
        "with SAVE_MODELS = True, or restore the committed saves from git."
    )

# Stop if a save was run on other data than the data loaded here
stale = {key: data_differences(save["config"]["run_on"]) for key, save in SAVES.items()}
stale = {key: lines for key, lines in stale.items() if lines}
if stale:
    raise ValueError(
        "The model saves were run on other data than data/smard.csv and the SMARD errors file:\n  "
        + "\n  ".join(f"{key}: {'; '.join(lines)}" for key, lines in stale.items())
        + "\nRe-run notebooks/API-connection.ipynb and forecast-metrics-claude.ipynb at the saves' "
        "resolution and extent, or refit and re-save the models in regression-models-claude.ipynb."
    )


def shared_setting(save, name):
    """A setting all saves must share: from `settings`, or the windows from `run_on`."""
    if name in save["config"]["settings"]:
        return save["config"]["settings"][name]
    return save["config"]["run_on"][name]


# Stop if the saves disagree in the forecast setting, the windows or the split methods
SHARED_SETTINGS = ["data_info", "windows", "split_methods", "validation", "test"]
reference_key = next(iter(SAVES))
for key, save in SAVES.items():
    for name in SHARED_SETTINGS:
        mine, theirs = shared_setting(save, name), shared_setting(SAVES[reference_key], name)
        if mine != theirs:
            raise ValueError(f"{key} and {reference_key} differ in {name}: {mine} vs. {theirs}")

overview = pd.DataFrame(
    {
        key: {
            "saved_at": save["config"]["saved_at"],
            "device": save["config"]["device"],
            "family": save["config"]["settings"]["family"],
            "architecture": save["config"]["settings"]["architecture"],
            "validation": save["config"]["run_on"]["validation"],
            "test": save["config"]["run_on"]["test"],
        }
        for key, save in SAVES.items()
    }
).T
print(f"{len(SAVES)} model saves, all run on the data loaded here")
display(overview)
for key, reason in SKIPPED.items():
    print(f"skipped {key}: {reason}")

In [ ]:
# The shared settings, taken from the saves (never typed in here)
SAVED_SETTINGS = SAVES[reference_key]["config"]["settings"]
DATA_INFO = {name: pd.Timedelta(value) for name, value in SAVED_SETTINGS["data_info"].items()}
SPLIT_METHODS = [method for method, on in SAVED_SETTINGS["split_methods"].items() if on]


def window_days(text):
    """'<first day> .. <last day>' -> the delivery days of that window."""
    first, last = (pd.Timestamp(part.strip()) for part in text.split(".."))
    return pd.date_range(first, last, freq="D")


VAL_DAYS = window_days(SAVES[reference_key]["config"]["run_on"]["validation"])
TEST_DAYS = window_days(SAVES[reference_key]["config"]["run_on"]["test"])
WINDOW_HOURS = {
    "validation": time_series.index[time_series.index.normalize().isin(VAL_DAYS)],
    "test": time_series.index[time_series.index.normalize().isin(TEST_DAYS)],
}
assert VAL_DAYS[-1] + pd.Timedelta(days=1) == TEST_DAYS[0], "validation must end the day before the test"

# Forecast setting per delivery day, as in spec 06 §2.1 (adaptive needs the cutoff)
DAYS = VAL_DAYS.append(TEST_DAYS)
FORECAST_SETTING = pd.DataFrame(index=DAYS)
FORECAST_SETTING["issue_time"] = DAYS - DATA_INFO["issue_days_before"] + DATA_INFO["issue_clock"]
FORECAST_SETTING["cutoff"] = FORECAST_SETTING["issue_time"] - DATA_INFO["actuals_lag"]


def available(stamps, cutoff):
    """True where a row stamped `stamps` (interval start) has ended by `cutoff`."""
    return stamps + RESOLUTION <= cutoff


cutoff_clock = DATA_INFO["issue_clock"] - DATA_INFO["actuals_lag"]
print("Settings taken from the saves")
print(f"  issue time    : {describe(DATA_INFO['issue_clock'])} on DAY−{DATA_INFO['issue_days_before'].days}, "
      f"actuals lag {describe(DATA_INFO['actuals_lag'])}, cutoff {describe(cutoff_clock)}")
print(f"  split methods : {SPLIT_METHODS}")
print(f"  windows       : train {SAVED_SETTINGS['windows']['train']}, "
      f"refit every {SAVED_SETTINGS['windows']['refit_every']}")
print(f"  validation    : {VAL_DAYS[0]:%Y-%m-%d} .. {VAL_DAYS[-1]:%Y-%m-%d} "
      f"({len(WINDOW_HOURS['validation']):,} hours)")
print(f"  test          : {TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d} "
      f"({len(WINDOW_HOURS['test']):,} hours)")

### 1.6 One forecast table per split method

`FORECASTS[split_method][window]` holds one column per forecast and one row per hour of the window.

- **Registry models**: their `{window}_{split_method}` run. The rolling ensemble therefore learns
  on `validation_rolling` and is scored on `test_rolling`; the static ensemble on
  `validation_static` and `test_static`, the pairing of spec 06's `CALIBRATED_ON`. A frozen model
  makes larger errors than a refitted one, so weights learned on rolling forecasts would suit the
  static members badly.
- **`smard`**: `fc_residual_load` from SMARD's errors file, the same in both split methods.
- **`seasonal_naive`**: the actual `residual_load` seven days earlier at the same local hour,
  looked up by timestamp (spec 06 Behaviour 19), the same in both split methods.

`ACTUAL` is the actual residual load per hour. `smard` and `seasonal_naive` are always built,
because the scoreboards need them; whether they are **members** is set in [§1.3](#sec-1-3).


In [ ]:
NAIVE_LAG = pd.Timedelta(days=7)
REGISTRY_MEMBERS = list(SAVES) if MEMBERS is None else list(MEMBERS)

for key in [key for key in REGISTRY_MEMBERS if key not in SAVES]:
    print(f"MEMBERS names {key!r}, but data/models/ has no complete save for it: dropped")
REGISTRY_MEMBERS = [key for key in REGISTRY_MEMBERS if key in SAVES]
MEMBER_KEYS = (
    REGISTRY_MEMBERS
    + (["smard"] if USE_SMARD_MEMBER else [])
    + (["seasonal_naive"] if USE_NAIVE_MEMBER else [])
)
assert len(MEMBER_KEYS) >= 2, f"an ensemble needs at least two members, got {MEMBER_KEYS}"

ACTUAL = time_series["residual_load"]
SMARD_FORECAST = smard_errors["fc_residual_load"]


def seasonal_naive(hours):
    """Actual residual load NAIVE_LAG earlier at the same local hour, looked up by timestamp."""
    return ACTUAL.reindex(hours - NAIVE_LAG).set_axis(hours)


FORECASTS = {}
for split_method in SPLIT_METHODS:
    FORECASTS[split_method] = {}
    for window, hours in WINDOW_HOURS.items():
        table = pd.DataFrame(index=hours)
        for key in SAVES:
            table[key] = SAVES[key]["runs"][f"{window}_{split_method}"].reindex(hours)
        table["smard"] = SMARD_FORECAST.reindex(hours)
        table["seasonal_naive"] = seasonal_naive(hours)
        FORECASTS[split_method][window] = table

summary = pd.DataFrame(
    {
        (split_method, window): table.notna().sum()
        for split_method, windows in FORECASTS.items()
        for window, table in windows.items()
    }
)
summary.columns.names = ["split method", "window"]
print(f"members ({len(MEMBER_KEYS)}): {', '.join(MEMBER_KEYS)}")
print("hours with a forecast, per column:")
display(summary)

### 1.7 Self-check


In [ ]:
# time_series is untouched
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert len(time_series) == LOADED["rows"]

# The configuration is consistent
assert set(METHODS) == {"equal_mean", "weighted", "regime_weighted", "adaptive", "stacking"}
assert any(method["enabled"] for method in METHODS.values()), "switch on at least one method"
assert set(REGIMES) == {"level", "season"} and any(REGIMES.values()), REGIMES
assert sorted(LEVEL_QUANTILES) == LEVEL_QUANTILES and all(0 < q < 1 for q in LEVEL_QUANTILES)
assert SHRINK_HOURS >= 0
assert all(isinstance(window, pd.Timedelta) for window in ADAPTIVE_WINDOWS)
assert 0 < INTERVAL["level"] < 1
assert isinstance(EXPORT_ENABLED, bool)

# Every member forecast is the save it came from, on exactly the window's hours
for split_method, windows in FORECASTS.items():
    for window, table in windows.items():
        assert table.index.equals(WINDOW_HOURS[window])
        for key in SAVES:
            saved = SAVES[key]["runs"][f"{window}_{split_method}"]
            assert saved.index.equals(WINDOW_HOURS[window]), f"{key}: {window} hours differ"
            assert table[key].equals(saved), f"{key}: {window}_{split_method} altered"

# The validation and test windows do not overlap
assert not WINDOW_HOURS["validation"].intersection(WINDOW_HOURS["test"]).size

print("setup self-check passed")
print(f"  {len(SAVES)} saves, {len(MEMBER_KEYS)} members, split methods {SPLIT_METHODS}")
print(f"  validation {len(WINDOW_HOURS['validation']):,} h, test {len(WINDOW_HOURS['test']):,} h, no overlap")

---

## <a id="sec-2"></a>2 The members compared

Which member wins where, built on the two team notebooks:

- the **five categories**, the **pick rule** and the plots of
  [`visualization-01-regression-best-models.ipynb`](visualization-01-regression-best-models.ipynb)
  (spec 09): `overall`, `ordinary`, `below_zero`, `low_extreme`, `high_extreme`
- the **risk flags** of
  [`visualization-02-classification-risk-labels.ipynb`](visualization-02-classification-risk-labels.ipynb)
  (spec 10), applied to every member

**One difference to spec 09:** spec 09 picks on the **test** year, which is right for a
presentation. Here every choice that feeds the ensemble is made on the **validation** year. The
test year is shown next to it **as a stability check only — used for no choice**: does the
validation winner also win the test year?

The risk flags are a diagnostic: they show which member catches which risk days. They do not feed
the weights.

### 2.1 Categories and scoreboards

Per window, the bin edges are the `{1, 25, 75, 99} %` quantiles of the **actual** over all
hours of that window, as in spec 06 §7.3 and spec 09 §1.3; `below_zero` is the fixed 0 MWh edge.
A row's MAE in a bin is computed twice: on the hours where the **actual** lies in the bin, and on
the hours where the row's **own forecast** does. **Forecast share** = hours the row forecasts in the
bin / hours the actual lies there (1.0: as often as it occurs; below 1: too rarely).

**Common hours** of a window: hours with an actual and a forecast from every member, SMARD and
seasonal naive, in both split methods.


In [ ]:
# Label and colour per row, copied from visualization-01-regression-best-models.ipynb
STYLE = {
    "sarimax_fourier": {"label": "SARIMAX + Fourier", "color": "#D9A53A"},
    "lgbm_direct": {"label": "LightGBM direct", "color": "#2C6EBA"},
    "lgbm_hybrid": {"label": "LightGBM hybrid", "color": "#2F8F5B"},
    "xgb_direct": {"label": "XGBoost direct", "color": "#E95D0F"},
    "xgb_hybrid": {"label": "XGBoost hybrid", "color": "#B10F0F"},
    "linear_direct": {"label": "Ridge direct", "color": "#7A4FA3"},
    "random_forest_hybrid": {"label": "Random forest hybrid", "color": "#8C564B"},
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A", "linestyle": "--"},
    "seasonal_naive": {"label": "Seasonal naive (DAY−7)", "color": "#9098A2", "linestyle": ":"},
}

# Bin regions on a residual-load axis, never a model's colour (visualization-01)
TAIL_COLOR = {
    "low_extreme": "#17BECF",   # cyan
    "below_zero": "#9EDAE5",    # light cyan
    "high_extreme": "#E6B800",  # gold
    "ordinary": "#E3E8EF",      # light grey
}

N_PICKS = 3          # top picks per category (spec 09)
MIN_BIN_SHARE = 0.5  # spec 09's forecast-share floor for the bin categories
WINDOW_ORDER = ["validation", "test"]
CATEGORIES = ["overall", "ordinary", "below_zero", "low_extreme", "high_extreme"]
BIN_CATEGORIES = CATEGORIES[1:]
TAIL_LEVELS = [0.01, 0.25, 0.75, 0.99]
ROW_KEYS = list(SAVES) + ["smard", "seasonal_naive"]  # every forecast in FORECASTS
DIAG_SPLIT = PLOT_SPLIT_METHOD if PLOT_SPLIT_METHOD in SPLIT_METHODS else SPLIT_METHODS[0]


def label_of(key):
    """Display label of a forecast key."""
    return STYLE.get(key, {"label": key})["label"]


def line_style(key):
    """Colour, line style and label of a forecast key for plt.plot."""
    style = STYLE.get(key, {"color": "black"})
    return {"color": style["color"], "linestyle": style.get("linestyle", "-"), "label": label_of(key)}


# Common hours per window: an actual and every forecast, in both split methods
COMMON = {}
for window, hours in WINDOW_HOURS.items():
    complete = ACTUAL.reindex(hours).notna()
    for split_method in SPLIT_METHODS:
        complete &= FORECASTS[split_method][window].notna().all(axis=1)
    COMMON[window] = hours[complete.to_numpy()]

# Edges over all hours of the window, as in spec 06 §7.3
EDGES = {window: ACTUAL[WINDOW_HOURS[window]].quantile(TAIL_LEVELS).to_numpy() for window in WINDOW_ORDER}


def bin_masks(values, window):
    """One mask per bin category, with the window's edges; below_zero overlaps low_extreme."""
    p1, p25, p75, p99 = EDGES[window]
    return {
        "ordinary": (values >= p25) & (values <= p75),
        "below_zero": values < LEVEL_ZERO,
        "low_extreme": values <= p1,
        "high_extreme": values > p99,
    }


def full_months(window):
    """Calendar months whose first and last day lie inside the window."""
    days = WINDOW_HOURS[window].normalize().unique()
    return pd.PeriodIndex(
        [m for m in days.to_period("M").unique() if m.start_time in days and m.end_time.normalize() in days]
    )


def window_scoreboard(split_method, window):
    """One row per forecast: accuracy, months beating SMARD and the four bins in both views."""
    hours = COMMON[window]
    actual = ACTUAL[hours]
    table = FORECASTS[split_method][window].loc[hours]
    errors = table.sub(actual, axis=0)  # forecast − actual
    smard_mae = errors["smard"].abs().mean()
    monthly = errors.abs().groupby(hours.to_period("M")).mean().loc[full_months(window)]
    by_actual = bin_masks(actual, window)

    rows = {}
    for key in ROW_KEYS:
        error = errors[key]
        row = {
            "MAE": error.abs().mean(),
            "RMSE": float(np.sqrt((error**2).mean())),
            "bias": error.mean(),
            "skill_pct": np.nan if key == "smard" else 100 * (1 - error.abs().mean() / smard_mae),
            "months_beating_smard": np.nan if key == "smard" else int((monthly[key] < monthly["smard"]).sum()),
        }
        by_forecast = bin_masks(table[key], window)
        for name in BIN_CATEGORIES:
            in_actual, in_forecast = by_actual[name], by_forecast[name]
            row[f"MAE_{name}_by_actual"] = error[in_actual].abs().mean()
            row[f"MAE_{name}_by_forecast"] = error[in_forecast].abs().mean()
            row[f"hours_{name}_by_actual"] = int(in_actual.sum())
            row[f"hours_{name}_by_forecast"] = int(in_forecast.sum())
            row[f"share_{name}"] = in_forecast.sum() / in_actual.sum() if in_actual.sum() else np.nan
        rows[key] = row
    return pd.DataFrame(rows).T


BOARDS = {(s, w): window_scoreboard(s, w) for s in SPLIT_METHODS for w in WINDOW_ORDER}

for window in WINDOW_ORDER:
    p1, p25, p75, p99 = EDGES[window]
    lost = len(WINDOW_HOURS[window]) - len(COMMON[window])
    print(
        f"{window:<10}: {len(COMMON[window]):,} common hours ({lost} lost), "
        f"{len(full_months(window))} full months; edges P1 {p1:,.0f} · P25 {p25:,.0f} · "
        f"P75 {p75:,.0f} · P99 {p99:,.0f} MWh"
    )

### 2.2 Member overview

Per split method: accuracy on the validation year and, as a check, on the test year. The skill is
against SMARD on the same hours (`100 · (1 − MAE / MAE_SMARD)`, positive = better than SMARD);
months beating SMARD counts the full calendar months of the window.


In [ ]:
OVERVIEW_COLUMNS = {
    "MAE": "MAE (MWh)",
    "bias": "bias (MWh)",
    "skill_pct": "skill vs SMARD (%)",
    "months_beating_smard": "months beating SMARD",
}

for split_method in SPLIT_METHODS:
    table = pd.concat(
        {window: BOARDS[(split_method, window)][list(OVERVIEW_COLUMNS)].rename(columns=OVERVIEW_COLUMNS)
         for window in WINDOW_ORDER},
        axis=1,
    )
    table = table.sort_values(("validation", "MAE (MWh)"))
    table.index = [label_of(key) for key in table.index]
    months = {window: len(full_months(window)) for window in WINDOW_ORDER}
    print(
        f"{split_method}: validation {len(COMMON['validation']):,} h / {months['validation']} full months, "
        f"test (check only) {len(COMMON['test']):,} h / {months['test']} full months; sorted by validation MAE"
    )
    display(table.style.format("{:,.1f}", na_rep="–", subset=[c for c in table.columns if "skill" in c[1]])
            .format("{:,.0f}", na_rep="–", subset=[c for c in table.columns if "skill" not in c[1]]))

### 2.3 Error correlation

Correlation of the hourly errors on the validation year. A combination can only gain where the
errors are not near-identical: two members with a correlation near 1 make the same mistakes.


In [ ]:
for split_method in SPLIT_METHODS:
    hours = COMMON["validation"]
    errors = FORECASTS[split_method]["validation"].loc[hours].sub(ACTUAL[hours], axis=0)
    correlation = errors[ROW_KEYS].corr()
    correlation.index = correlation.columns = [label_of(key) for key in ROW_KEYS]
    print(f"{split_method}: correlation of forecast − actual, validation year, {len(hours):,} h")
    display(correlation.style.format("{:.2f}").background_gradient(cmap="Blues", vmin=0.5, vmax=1))

### 2.4 Who wins where: spec 09's pick rule

The rule of spec 09 §2, applied to each window. Only registry members compete; SMARD and seasonal
naive never do. A member qualifies

- in `overall` with a skill vs SMARD above 0; ranked by months beating SMARD, then MAE
- in a bin when its MAE is below SMARD's **by actual and by forecast**, and its forecast share is at
  least `MIN_BIN_SHARE`; ranked by MAE by actual

The **validation** picks are the ones the ensemble may use. The **test** picks are the check; for
the `rolling` split they are spec 09's own picks, recomputed on the common hours of this notebook.


In [ ]:
CANDIDATES = REGISTRY_MEMBERS  # registry members only


def failed_rules(board, key, category):
    """The rules a candidate fails in a category; empty if it qualifies (spec 09 §2)."""
    if category == "overall":
        return [] if board.loc[key, "skill_pct"] > 0 else ["skill ≤ 0"]
    failed = []
    for view in ["actual", "forecast"]:
        metric = f"MAE_{category}_by_{view}"
        if not board.loc[key, metric] < board.loc["smard", metric]:
            failed.append(f"by {view}")
    share = board.loc[key, f"share_{category}"]
    if not share >= MIN_BIN_SHARE:
        failed.append(f"forecast share {share:.2f} < {MIN_BIN_SHARE}")
    return failed


def rank_key(board, key, category):
    """Sort key: best first, ties by model key."""
    if category == "overall":
        return (-board.loc[key, "months_beating_smard"], board.loc[key, "MAE"], key)
    return (board.loc[key, f"MAE_{category}_by_actual"], key)


def picks_of(board):
    """The qualifying candidates per category, best first."""
    return {
        category: sorted(
            (key for key in CANDIDATES if not failed_rules(board, key, category)),
            key=lambda key: rank_key(board, key, category),
        )
        for category in CATEGORIES
    }


PICKS = {(s, w): picks_of(BOARDS[(s, w)]) for s in SPLIT_METHODS for w in WINDOW_ORDER}


def shown(keys):
    """The first N_PICKS labels, joined; a dash if none qualifies."""
    return ", ".join(label_of(key) for key in keys[:N_PICKS]) or "– (none beats SMARD)"


for split_method in SPLIT_METHODS:
    rows = {}
    for category in CATEGORIES:
        validation, test = PICKS[(split_method, "validation")][category], PICKS[(split_method, "test")][category]
        same = "–" if not validation or not test else ("yes" if validation[0] == test[0] else "no")
        rows[category] = {
            "validation picks (choice)": shown(validation),
            "test picks (check only)": shown(test),
            "rank 1 the same": same,
        }
    print(f"{split_method}: top {N_PICKS} per category")
    display(pd.DataFrame(rows).T)

The MAE behind the picks, per bin and view, for the split method in `PLOT_SPLIT_METHOD`. Each
cell shows the validation value, the test value next to it is the check. A member that forecasts a
bin too rarely (forecast share below `MIN_BIN_SHARE`) cannot be picked there, however low its MAE by
actual.


In [ ]:
detail = {}
for name in BIN_CATEGORIES:
    for view in ["actual", "forecast"]:
        for window in WINDOW_ORDER:
            detail[(name, f"MAE by {view}", window)] = BOARDS[(DIAG_SPLIT, window)][f"MAE_{name}_by_{view}"]
    for window in WINDOW_ORDER:
        detail[(name, "forecast share", window)] = BOARDS[(DIAG_SPLIT, window)][f"share_{name}"]
detail = pd.DataFrame(detail).loc[ROW_KEYS]
detail.index = [label_of(key) for key in detail.index]
hours = {
    (name, window): BOARDS[(DIAG_SPLIT, window)].loc["smard", f"hours_{name}_by_actual"]
    for name in BIN_CATEGORIES for window in WINDOW_ORDER
}
print(f"{DIAG_SPLIT} split; hours in each bin by actual (validation / test):")
for name in BIN_CATEGORIES:
    print(f"  {name:<13}: {hours[(name, 'validation')]:,.0f} / {hours[(name, 'test')]:,.0f}")
display(
    detail.style.format("{:,.0f}", na_rep="–", subset=[c for c in detail.columns if c[1] != "forecast share"])
    .format("{:.2f}", na_rep="–", subset=[c for c in detail.columns if c[1] == "forecast share"])
)

### 2.5 Best single member

The reference every ensemble has to beat (spec 08 Behaviour 8): per split method, the registry
member with the lowest **validation** MAE. The member that happens to be best on the test year is
printed next to it as context; it is not the benchmark, because picking it would use the test year.


In [ ]:
BEST_MEMBER = {}
for split_method in SPLIT_METHODS:
    validation = BOARDS[(split_method, "validation")].loc[REGISTRY_MEMBERS, "MAE"]
    test = BOARDS[(split_method, "test")].loc[REGISTRY_MEMBERS, "MAE"]
    BEST_MEMBER[split_method] = validation.idxmin()
    best_test = test.idxmin()
    print(
        f"{split_method:<8}: best member {label_of(BEST_MEMBER[split_method])} "
        f"(validation MAE {validation.min():,.0f} MWh, test {test[BEST_MEMBER[split_method]]:,.0f} MWh)"
    )
    note = "the same member" if best_test == BEST_MEMBER[split_method] else "a different member"
    print(f"          best on test: {label_of(best_test)} ({test.min():,.0f} MWh), {note}")

### 2.6 Plots

All three for the split method in `PLOT_SPLIT_METHOD`, validation on the left, test (check only) on
the right.

**Monthly skill vs SMARD**: one point per full month; above 0 the member's MAE was lower than
SMARD's in that month (spec 09 §4.1).


In [ ]:
def skill_by(group_of, split_method, window):
    """Skill vs SMARD in % per group of common hours, one column per registry member."""
    hours = COMMON[window]
    mae = FORECASTS[split_method][window].loc[hours].sub(ACTUAL[hours], axis=0).abs().groupby(group_of(hours)).mean()
    return 100 * (1 - mae[REGISTRY_MEMBERS].div(mae["smard"], axis=0))


def skill_panels(group_of, title, xlabel, xvalues, keep=None):
    """Two panels (validation | test) of skill vs SMARD per member; `keep(window)` selects groups."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
    for ax, window in zip(axes, WINDOW_ORDER):
        skill = skill_by(group_of, DIAG_SPLIT, window)
        if keep is not None:
            skill = skill.loc[keep(window)]
        x = xvalues(skill.index)
        ax.axhline(0, linewidth=1.6, **line_style("smard"))
        for key in REGISTRY_MEMBERS:
            ax.plot(x, skill[key], marker="o", markersize=4, linewidth=1.4, **line_style(key))
        check = " (check only)" if window == "test" else ""
        ax.set_title(f"{window}{check}", fontsize=13)
        ax.set_xlabel(xlabel, color="grey")
        ax.grid(axis="y", color="0.9", linewidth=0.8)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.tick_params(colors="black", length=0)
    axes[0].set_ylabel("skill vs SMARD (%)", color="grey")
    axes[0].yaxis.set_major_formatter(lambda value, _: f"{value:+.0f} %")
    fig.suptitle(f"{title} · {DIAG_SPLIT} split", fontsize=15)
    axes[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
    fig.tight_layout()
    return fig, axes


# Full months only: the partial first and last month of a window stay out
fig, axes = skill_panels(
    lambda hours: hours.to_period("M"),
    "Monthly skill vs SMARD",
    "month",
    lambda index: index.to_timestamp(),
    keep=full_months,
)
for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%y"))
plt.show()

**Skill vs SMARD by hour of day**: one point per local hour, over every day of the window (e.g.
all 10:00 hours); above 0 the member is better than SMARD at that hour (spec 09 §4.3).


In [ ]:
fig, axes = skill_panels(lambda hours: hours.hour, "Skill vs SMARD by hour of day", "local hour", lambda index: index)
for ax in axes:
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlim(-0.5, 23.5)
plt.show()

**Error range per bin**: one bar per row on the hours where the **actual** lies in the bin. On
80 % of those hours the row's error (`forecast − actual`) lies inside its bar; the diamond is the
average error (bias). A shorter bar is a tighter forecast; a diamond right of 0 means the row
forecasts too high on average (spec 09 §5.2).


In [ ]:
RANGE_LEVEL = 0.8  # share of the hours inside each bar
tail = (1 - RANGE_LEVEL) / 2
shown_rows = REGISTRY_MEMBERS + ["smard"]

fig, axes = plt.subplots(len(BIN_CATEGORIES), 2, figsize=(16, 3.1 * len(BIN_CATEGORIES)), sharex="row")
for row_axes, name in zip(axes, BIN_CATEGORIES):
    for ax, window in zip(row_axes, WINDOW_ORDER):
        hours = COMMON[window]
        in_bin = bin_masks(ACTUAL[hours], window)[name]
        errors = FORECASTS[DIAG_SPLIT][window].loc[hours[in_bin.to_numpy()], shown_rows].sub(
            ACTUAL[hours[in_bin.to_numpy()]], axis=0
        )
        for y, key in enumerate(reversed(shown_rows)):
            low, high = errors[key].quantile([tail, 1 - tail])
            color = STYLE[key]["color"]
            ax.hlines(y, low, high, color=color, linewidth=7, alpha=0.8)
            ax.plot(errors[key].mean(), y, marker="D", color="white", markeredgecolor=color, markersize=7)
        ax.axvline(0, color="0.4", linewidth=1)
        ax.set_yticks(range(len(shown_rows)))
        ax.set_yticklabels([label_of(key) for key in reversed(shown_rows)] if window == "validation" else [])
        check = " (check only)" if window == "test" else ""
        ax.set_title(f"{name} · {window}{check} · {int(in_bin.sum()):,} h", fontsize=11,
                     color="black", backgroundcolor=TAIL_COLOR[name] + "55")
        ax.grid(axis="x", color="0.9", linewidth=0.8)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.tick_params(colors="black", length=0)
        ax.xaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
for ax in axes[-1]:
    ax.set_xlabel("forecast − actual (MWh)", color="grey")
fig.suptitle(f"Error range per bin (middle {RANGE_LEVEL:.0%}, diamond = bias) · {DIAG_SPLIT} split", fontsize=15)
fig.tight_layout()
plt.show()

### 2.7 Risk flags per member

The risk definition of [`risk-definition.ipynb`](../03_risk_classification/risk-definition.ipynb),
applied to every member's forecast with the **exported thresholds**, exactly as
[`visualization-02-classification-risk-labels.ipynb`](visualization-02-classification-risk-labels.ipynb)
§2 does for its two picks. The rule functions are copied from there; the thresholds are never
recomputed on a forecast.

| label | threshold | rules |
|---|---|---|
| high `rolling` | trailing 365-day P99 of the actual, per day | `any`, `3h` |
| low `rolling` | trailing 365-day P1 of the actual, per day | `any`, `3h` |
| low `zero` | 0 MWh | `any`, `3h` |

A day is **evaluable** when its actual labels are set and its common hours cover
`DAY_COMPLETENESS`; on any other day every flag is empty, never `False`. Against the actual's flags
each forecast day is a **hit** (both flagged), a **miss** (only the actual), a **false alarm** (only
the forecast) or quiet. `recall = hits / (hits + misses)`, `precision = hits / (hits + false
alarms)`; an empty cell is a zero denominator.


In [ ]:
LABELS_SOURCE = "notebooks/03_risk_classification/risk-definition.ipynb"
LABEL_FILES = {
    "daily": DATA_DIR / "risk_classification" / "risk_labels_daily.csv",
    "hourly": DATA_DIR / "risk_classification" / "risk_labels_hourly.csv",
}
STAMP = "%Y-%m-%d %H:%M:%S"
PERSISTENCE = pd.Timedelta("3h")  # the 3h rule, as a duration (risk-definition.ipynb §3.1)
MIN_RUN = n_obs(PERSISTENCE)
LABEL_PAIRS = [("high", "rolling"), ("low", "rolling"), ("low", "zero")]
RULES = ["any", "3h"]
DAILY_THRESHOLDS = [f"{d}_threshold_{b}" for d, b in LABEL_PAIRS]
DAILY_FLAGS = [f"{d}_risk_{b}_{r}" for d, b in LABEL_PAIRS for r in RULES]
HOURLY_FLAGS = [f"{d}_risk_hour_{b}" for d, b in LABEL_PAIRS]

for path in LABEL_FILES.values():
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Create it by running {LABELS_SOURCE}.")

# Label columns stay three-state: an empty cell is "not evaluable", never False
labels_daily = pd.read_csv(LABEL_FILES["daily"])
labels_daily["date"] = pd.to_datetime(labels_daily["date"], format="%Y-%m-%d")
labels_daily = labels_daily.set_index("date")[DAILY_THRESHOLDS + DAILY_FLAGS]
labels_daily[DAILY_FLAGS] = labels_daily[DAILY_FLAGS].astype("boolean")
labels_hourly = pd.read_csv(LABEL_FILES["hourly"])
labels_hourly["timestamp"] = pd.to_datetime(labels_hourly["timestamp"], format=STAMP)
labels_hourly = labels_hourly.set_index("timestamp")[["residual_load"] + HOURLY_FLAGS]
labels_hourly[HOURLY_FLAGS] = labels_hourly[HOURLY_FLAGS].astype("boolean")

# Stop if the labels come from another fetch of smard.csv
if not labels_hourly.index.equals(time_series.index) or not labels_hourly["residual_load"].equals(ACTUAL):
    raise ValueError(
        f"The risk labels do not match data/smard.csv. Re-run {LABELS_SOURCE} top to bottom."
    )


def is_set(flag):
    """True where a flag is set; False and empty both give False (visualization-02 §2.2)."""
    return pd.Series(flag.to_numpy(dtype=bool, na_value=False), index=flag.index)


def hourly_crossings(series, threshold_by_day, direction):
    """Per-observation crossing flags against each observation's own day's threshold.

    Returns nullable booleans: pd.NA wherever the day's threshold is undefined, so an undefined
    threshold can never masquerade as "not at risk" (visualization-02 §2.1).
    """
    threshold = pd.Series(threshold_by_day.reindex(series.index.normalize()).to_numpy(), index=series.index)
    crossed = (series >= threshold) if direction == "high" else (series <= threshold)
    return crossed.astype("boolean").mask(threshold.isna())


def day_rules(hour_flag, threshold_by_day, days, evaluable):
    """The `any` and `3h` day rules from per-observation crossing flags (visualization-02 §2.1).

    Consecutive rows count as consecutive, also across the missing spring-DST hour. Days with an
    undefined threshold, or not evaluable, are pd.NA in both rules rather than False.
    """
    day = pd.Series(hour_flag.index.normalize(), index=hour_flag.index)
    filled = hour_flag.fillna(False).astype(bool)
    any_rule = filled.groupby(day).any().reindex(days, fill_value=False)
    run_id = (filled != filled.shift()).cumsum()
    run_length = filled.groupby([day, run_id]).transform("size").where(filled, 0)
    persist_rule = (run_length.groupby(day).max() >= MIN_RUN).reindex(days, fill_value=False)
    valid = (threshold_by_day.notna() & evaluable).reindex(days, fill_value=False)
    return {
        "any": any_rule.astype("boolean").mask(~valid),
        "3h": persist_rule.astype("boolean").mask(~valid),
    }


def outcome(forecast, actual):
    """hit, miss, false alarm or quiet per row; empty where either flag is empty."""
    f, a = is_set(forecast), is_set(actual)
    labels = np.select([f & a, a & ~f, f & ~a], ["hit", "miss", "false alarm"], "quiet")
    return pd.Series(labels, index=actual.index).where(forecast.notna() & actual.notna())


def percent(part, whole):
    """part / whole in %, empty for a zero denominator."""
    return 100 * part / whole if whole else np.nan


# Evaluable days, thresholds and the actual's flags per window
LABEL_WINDOW = {}
for window in WINDOW_ORDER:
    hours = COMMON[window]
    days = WINDOW_HOURS[window].normalize().unique()
    per_day = pd.Series(1, index=hours).groupby(hours.normalize()).size().reindex(days, fill_value=0)
    evaluable = labels_daily.loc[days, DAILY_FLAGS].notna().all(axis=1) & per_day.ge(MIN_OBS_PER_DAY)
    hour_evaluable = evaluable.reindex(hours.normalize()).to_numpy()
    LABEL_WINDOW[window] = {
        "hours": hours,
        "days": days,
        "evaluable": evaluable,
        "hour_evaluable": hour_evaluable,
        "threshold": {(d, b): labels_daily.loc[days, f"{d}_threshold_{b}"] for d, b in LABEL_PAIRS},
        "actual_days": {
            (d, b, r): labels_daily.loc[days, f"{d}_risk_{b}_{r}"].mask(~evaluable)
            for d, b in LABEL_PAIRS for r in RULES
        },
        "actual_hours": {
            (d, b): labels_hourly.loc[hours, f"{d}_risk_hour_{b}"].mask(~hour_evaluable) for d, b in LABEL_PAIRS
        },
    }


def forecast_flags(series, window):
    """Hour and day flags of a forecast against the exported thresholds, per label pair."""
    w = LABEL_WINDOW[window]
    flags = {}
    for d, b in LABEL_PAIRS:
        hour = hourly_crossings(series, w["threshold"][(d, b)], d).mask(~w["hour_evaluable"])
        flags[(d, b)] = {"hour": hour, **day_rules(hour, w["threshold"][(d, b)], w["days"], w["evaluable"])}
    return flags


# Reproduction check: the rules applied to the actual reproduce the label files on every
# evaluable day whose hours are all common (visualization-02 §2.3)
for window in WINDOW_ORDER:
    w = LABEL_WINDOW[window]
    actual_flags = forecast_flags(ACTUAL[w["hours"]], window)
    hours_per_day = pd.Series(1, index=WINDOW_HOURS[window]).groupby(WINDOW_HOURS[window].normalize()).size()
    common_per_day = pd.Series(1, index=w["hours"]).groupby(w["hours"].normalize()).size()
    fully_common = common_per_day.index[common_per_day.eq(hours_per_day.reindex(common_per_day.index))]
    for d, b in LABEL_PAIRS:
        for r in RULES:
            mine = actual_flags[(d, b)][r].reindex(fully_common)
            theirs = w["actual_days"][(d, b, r)].reindex(fully_common)
            same = (mine.isna() & theirs.isna()) | (mine.fillna(False) == theirs.fillna(False)) & mine.notna() & theirs.notna()
            assert same.all(), f"{window} {d} {b} {r}: rule and label file disagree on {(~same).sum()} days"

# Scores per window, split method and forecast
records = []
for window in WINDOW_ORDER:
    w = LABEL_WINDOW[window]
    for split_method in SPLIT_METHODS:
        for key in ROW_KEYS:
            if key in ("smard", "seasonal_naive") and split_method != SPLIT_METHODS[0]:
                continue  # the same in both split methods: scored once
            flags = forecast_flags(FORECASTS[split_method][window].loc[w["hours"], key], window)
            for d, b in LABEL_PAIRS:
                for r in RULES:
                    days = outcome(flags[(d, b)][r], w["actual_days"][(d, b, r)])
                    hit, miss, false_alarm = (int(days.eq(name).sum()) for name in ["hit", "miss", "false alarm"])
                    records.append({
                        "window": window,
                        "split_method": "none" if key in ("smard", "seasonal_naive") else split_method,
                        "member": key,
                        "label": f"{d} {b} {r}",
                        "actual days": hit + miss,
                        "hit": hit,
                        "miss": miss,
                        "false alarm": false_alarm,
                        "recall %": percent(hit, hit + miss),
                        "precision %": percent(hit, hit + false_alarm),
                    })
RISK_SCORES = pd.DataFrame(records)

for window in WINDOW_ORDER:
    scores = RISK_SCORES[RISK_SCORES["window"].eq(window)]
    actual_days = scores.groupby("label", sort=False)["actual days"].first()
    table = scores.pivot_table(
        index=["split_method", "member"], columns="label", values=["recall %", "precision %"], sort=False
    ).swaplevel(axis=1)
    table = table.reindex(columns=pd.MultiIndex.from_product([actual_days.index, ["recall %", "precision %"]]))
    order = [(s, k) for s in SPLIT_METHODS for k in REGISTRY_MEMBERS] + [("none", "smard"), ("none", "seasonal_naive")]
    table = table.reindex([row for row in order if row in table.index])
    table.index = [f"{label_of(k)} ({s})" for s, k in table.index]
    check = " (check only)" if window == "test" else ""
    evaluable = LABEL_WINDOW[window]["evaluable"]
    print(f"{window}{check}: {int(evaluable.sum())} evaluable days of {len(evaluable)}; actual days per label: "
          + ", ".join(f"{label} {count}" for label, count in actual_days.items()))
    display(table.style.format("{:.0f}", na_rep="–").background_gradient(cmap="Greens", vmin=0, vmax=100))

### 2.8 Self-check


In [ ]:
# time_series is untouched
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Common hours lie inside their windows; every forecast is present on them
for window in WINDOW_ORDER:
    assert COMMON[window].isin(WINDOW_HOURS[window]).all()
    for split_method in SPLIT_METHODS:
        assert FORECASTS[split_method][window].loc[COMMON[window]].notna().all().all()

# SMARD's MAE equals a direct recomputation from smard_forecast_errors_hourly.csv
for (split_method, window), board in BOARDS.items():
    direct = smard_errors.loc[COMMON[window], "err_residual_load"].abs().mean()
    assert np.isclose(board.loc["smard", "MAE"], direct), (split_method, window)

# Picks come from the registry members only, and each pick passes every rule
for (split_method, window), picks in PICKS.items():
    for category, keys in picks.items():
        assert set(keys) <= set(CANDIDATES)
        assert all(not failed_rules(BOARDS[(split_method, window)], key, category) for key in keys)

# The best member is chosen on the validation year
for split_method, key in BEST_MEMBER.items():
    assert key == BOARDS[(split_method, "validation")].loc[REGISTRY_MEMBERS, "MAE"].idxmin()

# Risk scores: hits + misses equal the actual's flagged evaluable days of every label
for window in WINDOW_ORDER:
    for d, b in LABEL_PAIRS:
        for r in RULES:
            flagged = int(is_set(LABEL_WINDOW[window]["actual_days"][(d, b, r)]).sum())
            counts = RISK_SCORES[(RISK_SCORES["window"] == window) & (RISK_SCORES["label"] == f"{d} {b} {r}")]
            assert counts["actual days"].eq(flagged).all(), (window, d, b, r)

print("section 2 self-check passed")
print("  common hours complete, SMARD re-scored from its file, picks follow spec 09's rule,")
print("  best member chosen on validation, risk rules reproduce the label files")

---

## <a id="sec-3"></a>3 Combination methods

Every method turns the members' forecasts for one hour into one ensemble forecast. Its parameters
(weights, regime edges, the adaptive window, the meta-model) are fitted on the **validation year
only**, separately per split method (§1.6). The test-year forecasts are built here
but **scored in §5, once**.

Two rules hold for every method:

- **An hour with a missing member forecast gets no ensemble forecast.** The ensemble never
  re-weights over the remaining members, because that would be a fallback (spec 06 Behaviour 21).
- The validation forecasts below are **in-sample** for every method with fitted parameters: the
  weights were chosen on the same hours. §4 adds the honest out-of-fold version for the bands.

### 3.1 The member matrix and the regimes

`FIT_HOURS[split_method]`: the validation hours with an actual and every member's forecast, the
only hours any parameter is fitted on.

**Regimes** (for `regime_weighted`) come from the **members' mean forecast** of an hour, which is
known at issue time; the actual is not. The low edge is the fixed 0 MWh (`LEVEL_ZERO`), as spec 06's
`below_zero` bin: hours below 0 MWh grow from year to year, so a quantile edge would mean something
different in the test year. The further edges are the `LEVEL_QUANTILES` of the members' mean
forecast on the fit hours. The bins are half-open: a value on an edge belongs to the bin above it.


In [ ]:
from scipy import sparse

METHOD_KEYS = [key for key, method in METHODS.items() if method["enabled"]]
MEMBER_TABLE = {
    split_method: {window: FORECASTS[split_method][window][MEMBER_KEYS] for window in WINDOW_ORDER}
    for split_method in SPLIT_METHODS
}
FIT_HOURS = {}
for split_method in SPLIT_METHODS:
    table = MEMBER_TABLE[split_method]["validation"]
    complete = table.notna().all(axis=1) & ACTUAL.reindex(table.index).notna()
    FIT_HOURS[split_method] = table.index[complete.to_numpy()]


def member_mean(table):
    """The members' mean forecast per hour; empty if a member is missing."""
    return table.mean(axis=1, skipna=False)


# Regime edges: 0 MWh, then quantiles of the members' mean forecast on the fit hours
REGIME_EDGES = {}
for split_method in SPLIT_METHODS:
    fit = MEMBER_TABLE[split_method]["validation"].loc[FIT_HOURS[split_method]]
    quantiles = member_mean(fit).quantile(LEVEL_QUANTILES).to_numpy()
    edges = np.r_[LEVEL_ZERO, quantiles]
    assert (np.diff(edges) > 0).all(), f"{split_method}: regime edges not increasing: {edges}"
    REGIME_EDGES[split_method] = edges

percent_names = [f"P{100 * q:g}" for q in LEVEL_QUANTILES]
LEVEL_NAMES = (
    ["below_zero", f"0..{percent_names[0]}"]
    + [f"{low}..{high}" for low, high in zip(percent_names, percent_names[1:])]
    + [f"> {percent_names[-1]}"]
)
if LEVEL_QUANTILES == [0.25, 0.75, 0.99]:  # the default edges carry the names of spec 08
    LEVEL_NAMES = ["below_zero", "low", "ordinary", "high", "high_extreme"]
REGIME_NAMES = (
    [f"{level} / {season}" for level in LEVEL_NAMES for season in SEASON_ORDER]
    if REGIMES["season"] and REGIMES["level"]
    else SEASON_ORDER if REGIMES["season"] else LEVEL_NAMES
)


def regime_of(table, split_method):
    """The regime of each hour from the members' mean forecast; empty if a member is missing."""
    mean = member_mean(table)
    level = pd.Series(
        np.array(LEVEL_NAMES)[np.searchsorted(REGIME_EDGES[split_method], mean.fillna(0).to_numpy(), side="right")],
        index=table.index,
    )
    season = pd.Series(table.index.month.map(SEASON_OF_MONTH), index=table.index)
    if REGIMES["level"] and REGIMES["season"]:
        regime = level + " / " + season
    elif REGIMES["season"]:
        regime = season
    else:
        regime = level
    return regime.where(mean.notna())


for split_method in SPLIT_METHODS:
    edges = ", ".join(f"{value:,.0f}" for value in REGIME_EDGES[split_method])
    print(f"{split_method:<8}: {len(FIT_HOURS[split_method]):,} fit hours; level edges {edges} MWh "
          f"(0 MWh, then {', '.join(percent_names)} of the members' mean forecast)")
print(f"regimes: {REGIME_NAMES}")

**Who wins where, by regime.** The validation MAE of each member in each regime, the members'
mean forecast deciding the regime as it will for `regime_weighted`; the test year next to it is the
check only. This is the table spec 08 Behaviour 7 placed in the diagnostic, moved here because the
regimes are defined here.


In [ ]:
def regime_mae(split_method, window):
    """MAE per member and regime, with the hours per regime."""
    table = MEMBER_TABLE[split_method][window]
    hours = table.index[(table.notna().all(axis=1) & ACTUAL.reindex(table.index).notna()).to_numpy()]
    regime = regime_of(table.loc[hours], split_method)
    errors = table.loc[hours].sub(ACTUAL[hours], axis=0).abs()
    mae = errors.groupby(regime).mean().reindex(REGIME_NAMES)
    return mae, regime.value_counts().reindex(REGIME_NAMES, fill_value=0)


for split_method in SPLIT_METHODS:
    rows = {}
    for window in WINDOW_ORDER:
        mae, count = regime_mae(split_method, window)
        best = mae.idxmin(axis=1)
        for regime in REGIME_NAMES:
            rows.setdefault(regime, {})
            rows[regime][(window, "hours")] = int(count[regime])
            rows[regime][(window, "best member")] = label_of(best[regime]) if pd.notna(best[regime]) else "–"
            rows[regime][(window, "its MAE")] = mae.loc[regime].min()
            rows[regime][(window, "SMARD MAE")] = mae.loc[regime, "smard"] if "smard" in mae else np.nan
    table = pd.DataFrame(rows).T
    table[("check", "same best member")] = np.where(
        table[("validation", "best member")] == table[("test", "best member")], "yes", "no"
    )
    print(f"{split_method}: best member per regime (validation = choice, test = check only)")
    display(table.style.format("{:,.0f}", na_rep="–", subset=[c for c in table.columns if "MAE" in c[1] or c[1] == "hours"]))

mae, count = regime_mae(DIAG_SPLIT, "validation")
mae.columns = [label_of(key) for key in mae.columns]
print(f"{DIAG_SPLIT}: validation MAE (MWh) per member and regime")
display(mae.style.format("{:,.0f}", na_rep="–").highlight_min(axis=1, color="#D7EBD0"))

### 3.2 `equal_mean`

The plain mean of the members. No parameters, so its validation forecast is already out-of-sample,
and it is the floor every fitted method has to beat. Equal weights are known to be hard to beat
(the "forecast combination puzzle").


In [ ]:
ENSEMBLE = {split_method: {window: pd.DataFrame(index=WINDOW_HOURS[window]) for window in WINDOW_ORDER}
            for split_method in SPLIT_METHODS}
WEIGHT_RECORDS = []  # method, split_method, regime, member, weight (the weights export, §6)
COST = {}  # (method, split_method) -> seconds: fit (§3), test forecast (§3), band calibration (§4)


def add_cost(method, split_method, **seconds):
    """Add measured seconds to a method's fit, forecast or calibration (Scoreboard A, §5.2)."""
    entry = COST.setdefault((method, split_method), {"fit": 0.0, "forecast": 0.0, "calibration": 0.0})
    for part, value in seconds.items():
        entry[part] += value


def record_weights(method, split_method, regime, weights):
    """Store one weight vector (members in MEMBER_KEYS order)."""
    for member, weight in zip(MEMBER_KEYS, weights):
        WEIGHT_RECORDS.append({"method": method, "split_method": split_method, "regime": regime,
                               "member": member, "weight": float(weight)})


def combine(table, weights):
    """Weighted sum per hour; empty wherever a member is missing (no re-weighting)."""
    values = table[MEMBER_KEYS].to_numpy()
    return pd.Series((values * weights).sum(axis=1), index=table.index).where(table[MEMBER_KEYS].notna().all(axis=1))


if "equal_mean" in METHOD_KEYS:
    equal = np.full(len(MEMBER_KEYS), 1 / len(MEMBER_KEYS))
    for split_method in SPLIT_METHODS:
        record_weights("equal_mean", split_method, "all", equal)
        for window in WINDOW_ORDER:
            started = time.perf_counter()
            ENSEMBLE[split_method][window]["equal_mean"] = combine(MEMBER_TABLE[split_method][window], equal)
            if window == "test":
                add_cost("equal_mean", split_method, forecast=time.perf_counter() - started)
    print(f"equal_mean: weight 1/{len(MEMBER_KEYS)} = {1 / len(MEMBER_KEYS):.3f} for each of {', '.join(MEMBER_KEYS)}")
else:
    print("equal_mean switched off in METHODS")

### 3.3 `weighted`

One weight per member, `w ≥ 0` and `Σ w = 1`, chosen to minimise the **MAE** on the fit hours.
MAE with these constraints is a linear programme, solved exactly by `scipy.optimize.linprog`
(HiGHS): minimise `Σ u_t` subject to `u_t ≥ |actual_t − Σ w_i · forecast_it|`. With near-identical
members the optimum is not unique: read the combined forecast, never one weight on its own.


In [ ]:
def lp_weights(forecasts, actual):
    """Weights w >= 0, sum 1, minimising the MAE of forecasts @ w against actual (an exact LP)."""
    n, m = forecasts.shape
    if n == 0:
        return None
    x = sparse.csr_matrix(forecasts)
    eye = sparse.identity(n, format="csr")
    a_ub = sparse.vstack([sparse.hstack([-x, -eye]), sparse.hstack([x, -eye])]).tocsr()
    b_ub = np.r_[-actual, actual]
    cost = np.r_[np.zeros(m), np.ones(n)]
    a_eq = sparse.csr_matrix(np.r_[np.ones(m), np.zeros(n)].reshape(1, -1))
    result = linprog(
        cost, A_ub=a_ub, b_ub=b_ub, A_eq=a_eq, b_eq=[1.0],
        bounds=[(0, 1)] * m + [(0, None)] * n, method="highs",
    )
    if not result.success:
        raise RuntimeError(f"weight LP failed: {result.message}")
    weights = np.clip(result.x[:m], 0, None)
    return weights / weights.sum()


GLOBAL_WEIGHTS, GLOBAL_SECONDS = {}, {}
for split_method in SPLIT_METHODS:
    started = time.perf_counter()
    fit = MEMBER_TABLE[split_method]["validation"].loc[FIT_HOURS[split_method]]
    GLOBAL_WEIGHTS[split_method] = lp_weights(fit.to_numpy(), ACTUAL[FIT_HOURS[split_method]].to_numpy())
    GLOBAL_SECONDS[split_method] = time.perf_counter() - started

if "weighted" in METHOD_KEYS:
    for split_method in SPLIT_METHODS:
        record_weights("weighted", split_method, "all", GLOBAL_WEIGHTS[split_method])
        add_cost("weighted", split_method, fit=GLOBAL_SECONDS[split_method])
        for window in WINDOW_ORDER:
            started = time.perf_counter()
            ENSEMBLE[split_method][window]["weighted"] = combine(MEMBER_TABLE[split_method][window], GLOBAL_WEIGHTS[split_method])
            if window == "test":
                add_cost("weighted", split_method, forecast=time.perf_counter() - started)
    weights = pd.DataFrame(GLOBAL_WEIGHTS, index=[label_of(key) for key in MEMBER_KEYS])
    print("weighted: weights per split method (fitted on the validation year)")
    display(weights.style.format("{:.3f}").bar(color="#F4B6B6", vmin=0, vmax=1))
else:
    print("weighted switched off in METHODS (its weights are still fitted: regime_weighted shrinks toward them)")

### 3.4 `regime_weighted`

The same LP per regime, on the fit hours of that regime. A regime with few hours has noisy
weights, so each regime's weights are **shrunk toward the global weights** of §3.3:

`w_regime = n / (n + k) · w_fit + k / (n + k) · w_global`, with `n` the regime's fit hours and
`k = SHRINK_HOURS`.

A regime with no fit hours therefore gets the global weights; this follows from the formula, it is
no fallback. The table shows `n` and the share `n / (n + k)` of the regime's own fit.


In [ ]:
REGIME_WEIGHTS, REGIME_SECONDS = {}, {}
for split_method in SPLIT_METHODS:
    started = time.perf_counter()
    fit = MEMBER_TABLE[split_method]["validation"].loc[FIT_HOURS[split_method]]
    actual = ACTUAL[FIT_HOURS[split_method]]
    regime = regime_of(fit, split_method)
    REGIME_WEIGHTS[split_method] = {}
    for name in REGIME_NAMES:
        in_regime = (regime == name).to_numpy()
        n = int(in_regime.sum())
        own = lp_weights(fit.to_numpy()[in_regime], actual.to_numpy()[in_regime]) if n else GLOBAL_WEIGHTS[split_method]
        share = n / (n + SHRINK_HOURS) if n + SHRINK_HOURS > 0 else 1.0
        REGIME_WEIGHTS[split_method][name] = {
            "hours": n,
            "own share": share,
            "weights": share * own + (1 - share) * GLOBAL_WEIGHTS[split_method],
        }
    REGIME_SECONDS[split_method] = GLOBAL_SECONDS[split_method] + time.perf_counter() - started


def regime_combine(table, split_method):
    """Each hour combined with the weights of its regime; empty where a member is missing."""
    regime = regime_of(table, split_method)
    weights = np.vstack([
        REGIME_WEIGHTS[split_method][name]["weights"] if pd.notna(name) else np.full(len(MEMBER_KEYS), np.nan)
        for name in regime
    ])
    values = table[MEMBER_KEYS].to_numpy()
    return pd.Series((values * weights).sum(axis=1), index=table.index).where(table[MEMBER_KEYS].notna().all(axis=1))


if "regime_weighted" in METHOD_KEYS:
    for split_method in SPLIT_METHODS:
        for name, entry in REGIME_WEIGHTS[split_method].items():
            record_weights("regime_weighted", split_method, name, entry["weights"])
        add_cost("regime_weighted", split_method, fit=REGIME_SECONDS[split_method])
        for window in WINDOW_ORDER:
            started = time.perf_counter()
            ENSEMBLE[split_method][window]["regime_weighted"] = regime_combine(MEMBER_TABLE[split_method][window], split_method)
            if window == "test":
                add_cost("regime_weighted", split_method, forecast=time.perf_counter() - started)
        table = pd.DataFrame(
            {name: {"fit hours": entry["hours"], "own share": entry["own share"],
                    **{label_of(key): w for key, w in zip(MEMBER_KEYS, entry["weights"])}}
             for name, entry in REGIME_WEIGHTS[split_method].items()}
        ).T
        print(f"{split_method}: regime weights after shrinkage (SHRINK_HOURS = {SHRINK_HOURS})")
        member_columns = [label_of(key) for key in MEMBER_KEYS]
        display(
            table.style.format("{:,.0f}", subset=["fit hours"]).format("{:.0%}", subset=["own share"])
            .format("{:.2f}", subset=member_columns).background_gradient(cmap="Greens", subset=member_columns, vmin=0, vmax=1)
        )
else:
    print("regime_weighted switched off in METHODS")

### 3.5 `adaptive`

Weights that follow recent performance. For delivery day `DAY`, a member's weight is proportional
to `1 / MAE`, the MAE taken over the trailing window of hours whose **error is known at `DAY`'s
availability cutoff** (`t + resolution ≤ cutoff`, the spec 06 rule with the saves' `DATA_INFO`).

- The trailing errors come from the members' forecasts of the same split method; at the start of
  the test year the window reaches back into the validation year, whose errors are published long
  before.
- A day whose window would reach before the first validation hour has no adaptive forecast: the
  saves hold no earlier forecasts, and an equal-weight stand-in would be a fallback. Only the first
  validation days are affected, never the test year.
- The window is chosen from `ADAPTIVE_WINDOWS` by the MAE on the validation days where every
  candidate has a forecast, then frozen.

Using test-year actuals as they are published is an **online update, not selection**: the weights
for `DAY` never see an actual after `DAY`'s cutoff. The closing self-check tests this by truncation.


In [ ]:
def adaptive_run(split_method, length):
    """Adaptive forecasts for every validation and test day, and the weights per day."""
    table = pd.concat([MEMBER_TABLE[split_method][window] for window in WINDOW_ORDER])
    stamps = table.index
    absolute = table.sub(ACTUAL.reindex(stamps), axis=0).abs()
    values = np.vstack([np.zeros(len(MEMBER_KEYS)), np.cumsum(absolute.fillna(0).to_numpy(), axis=0)])
    counts = np.vstack([np.zeros(len(MEMBER_KEYS)), np.cumsum(absolute.notna().to_numpy(), axis=0)])
    day_of = stamps.normalize()

    forecast = pd.Series(np.nan, index=stamps)
    weights_by_day = {}
    for day in DAYS:
        cutoff = FORECAST_SETTING.at[day, "cutoff"]
        start = cutoff - length
        if start < stamps[0]:
            continue  # the window reaches before the saved forecasts: no adaptive forecast
        low = stamps.searchsorted(start, side="left")                  # stamps >= start
        high = stamps.searchsorted(cutoff - RESOLUTION, side="right")  # stamp + resolution <= cutoff
        mae = (values[high] - values[low]) / (counts[high] - counts[low])
        inverse = 1 / mae
        weights = inverse / inverse.sum()
        weights_by_day[day] = weights
        in_day = day_of == day
        forecast[in_day] = combine(table.loc[in_day], weights).to_numpy()
    return forecast, pd.DataFrame(weights_by_day, index=MEMBER_KEYS).T


ADAPTIVE = {}
if "adaptive" in METHOD_KEYS:
    for split_method in SPLIT_METHODS:
        started = time.perf_counter()
        runs = {length: adaptive_run(split_method, length) for length in ADAPTIVE_WINDOWS}
        hours = FIT_HOURS[split_method]
        defined = pd.concat({length: run[0].reindex(hours) for length, run in runs.items()}, axis=1).notna().all(axis=1)
        scored = hours[defined.to_numpy()]
        mae = {length: (run[0][scored] - ACTUAL[scored]).abs().mean() for length, run in runs.items()}
        chosen = min(mae, key=mae.get)
        add_cost("adaptive", split_method, fit=time.perf_counter() - started)  # incl. the test forecast
        forecast, weights = runs[chosen]
        ADAPTIVE[split_method] = {"window": chosen, "weights": weights, "validation_mae": mae, "scored_hours": len(scored)}
        for window in WINDOW_ORDER:
            ENSEMBLE[split_method][window]["adaptive"] = forecast.reindex(WINDOW_HOURS[window])
        for day, row in weights.iterrows():
            record_weights("adaptive", split_method, f"{day:%Y-%m-%d}", row.to_numpy())
        candidates = ", ".join(f"{describe(length)} {value:,.0f}" for length, value in mae.items())
        missing_days = len(VAL_DAYS) - weights.index.isin(VAL_DAYS).sum()
        print(f"{split_method:<8}: window {describe(chosen)} chosen (validation MAE, {len(scored):,} h: {candidates} MWh); "
              f"{missing_days} first validation days without a forecast")
else:
    print("adaptive switched off in METHODS")

The adaptive weights over time for the split method in `PLOT_SPLIT_METHOD`: one band per member,
stacked to 1. The dashed line marks the start of the test year.


In [ ]:
if DIAG_SPLIT in ADAPTIVE:
    weights = ADAPTIVE[DIAG_SPLIT]["weights"]
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.stackplot(
        weights.index, weights.T.to_numpy(),
        labels=[label_of(key) for key in MEMBER_KEYS],
        colors=[STYLE.get(key, {"color": "grey"})["color"] for key in MEMBER_KEYS], alpha=0.85,
    )
    ax.axvline(TEST_DAYS[0], color="black", linestyle="--", linewidth=1.2)
    ax.text(TEST_DAYS[0], 0.97, " test year (check only) →", fontsize=10, va="top", color="white", fontweight="bold")
    ax.set_ylim(0, 1)
    style_timeseries(ax, f"Adaptive weights per day · window {describe(ADAPTIVE[DIAG_SPLIT]['window'])} · {DIAG_SPLIT} split", "weight")
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
    ax.xaxis.set_major_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
    plt.show()
else:
    print("adaptive switched off in METHODS")

### 3.6 `stacking`

A `Ridge` meta-model on the member forecasts plus the level regime as one-hot columns, fitted on the
fit hours, with the fixed `alpha` of its `METHODS` entry. Off by default: it has the most parameters,
overlaps `regime_weighted`, and with few hours per regime it is the most likely to overfit.


In [ ]:
STACKING = {}


def stacking_design(table, split_method):
    """Member forecasts plus one-hot regimes; rows with a missing member stay empty."""
    regime = regime_of(table, split_method)
    one_hot = pd.get_dummies(regime).reindex(columns=REGIME_NAMES, fill_value=False).astype(float)
    return pd.concat([table[MEMBER_KEYS], one_hot], axis=1)


if "stacking" in METHOD_KEYS:
    for split_method in SPLIT_METHODS:
        started = time.perf_counter()
        design = stacking_design(MEMBER_TABLE[split_method]["validation"].loc[FIT_HOURS[split_method]], split_method)
        model = Ridge(alpha=METHODS["stacking"]["alpha"]).fit(design, ACTUAL[FIT_HOURS[split_method]])
        STACKING[split_method] = model
        add_cost("stacking", split_method, fit=time.perf_counter() - started)
        for window in WINDOW_ORDER:
            started = time.perf_counter()
            table = MEMBER_TABLE[split_method][window]
            design = stacking_design(table, split_method)
            complete = table.notna().all(axis=1)
            forecast = pd.Series(np.nan, index=table.index)
            forecast[complete] = model.predict(design[complete])
            ENSEMBLE[split_method][window]["stacking"] = forecast
            if window == "test":
                add_cost("stacking", split_method, forecast=time.perf_counter() - started)
        coefficients = pd.Series(model.coef_, index=[label_of(c) if c in STYLE else c for c in design.columns])
        print(f"{split_method}: stacking coefficients (intercept {model.intercept_:,.0f} MWh)")
        display(coefficients.to_frame("coefficient").style.format("{:.3f}"))
else:
    print("stacking switched off in METHODS (set METHODS['stacking']['enabled'] = True to try it)")

### 3.7 First look: validation MAE (in-sample)

The validation MAE of every enabled method next to the best single member ([§2.5](#sec-2)) and
SMARD. **This is in-sample** for every fitted method, so it flatters them; it only shows that the
methods work. The honest comparison is the test year in §5.


In [ ]:
for split_method in SPLIT_METHODS:
    hours = COMMON["validation"]
    rows = {}
    for key in METHOD_KEYS:
        forecast = ENSEMBLE[split_method]["validation"][key].reindex(hours)
        scored = forecast.dropna().index
        rows[METHODS[key]["label"]] = {"MAE (MWh)": (forecast[scored] - ACTUAL[scored]).abs().mean(),
                                       "hours": len(scored),
                                       "in-sample": "no" if key == "equal_mean" else "yes"}
    best = BEST_MEMBER[split_method]
    for key, note in [(best, "best member"), ("smard", "benchmark")]:
        forecast = FORECASTS[split_method]["validation"].loc[hours, key]
        rows[f"{label_of(key)} ({note})"] = {"MAE (MWh)": (forecast - ACTUAL[hours]).abs().mean(),
                                             "hours": len(hours), "in-sample": "–"}
    table = pd.DataFrame(rows).T
    print(f"{split_method}: validation year")
    display(table.style.format({"MAE (MWh)": "{:,.0f}", "hours": "{:,.0f}"}))

### 3.8 Self-check


In [ ]:
# time_series is untouched
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Every parameter is fitted on validation hours only
for split_method in SPLIT_METHODS:
    assert FIT_HOURS[split_method].isin(WINDOW_HOURS["validation"]).all()
    assert not FIT_HOURS[split_method].isin(WINDOW_HOURS["test"]).any()

# Weights are >= 0 and sum to 1
weights = pd.DataFrame(WEIGHT_RECORDS)
if len(weights):
    sums = weights.groupby(["method", "split_method", "regime"])["weight"].sum()
    assert np.allclose(sums, 1, atol=1e-9), sums[~np.isclose(sums, 1, atol=1e-9)]
    assert (weights["weight"] >= -1e-12).all()

# equal_mean is the row mean; every ensemble is empty exactly where a member is empty
for split_method in SPLIT_METHODS:
    for window in WINDOW_ORDER:
        members = MEMBER_TABLE[split_method][window]
        missing = members.isna().any(axis=1)
        if "equal_mean" in METHOD_KEYS:
            assert np.allclose(ENSEMBLE[split_method][window]["equal_mean"].dropna(), members.mean(axis=1)[~missing])
        for key in METHOD_KEYS:
            forecast = ENSEMBLE[split_method][window][key]
            assert forecast[missing].isna().all(), f"{key}: a forecast where a member is missing"
            if key != "adaptive":
                assert forecast[~missing].notna().all(), f"{key}: empty although every member exists"

# Adaptive: the test year is fully covered, and every weight row reads only errors before the cutoff
for split_method, entry in ADAPTIVE.items():
    assert entry["weights"].index.isin(TEST_DAYS).sum() == len(TEST_DAYS)
    assert ENSEMBLE[split_method]["test"]["adaptive"].notna().sum() == (~MEMBER_TABLE[split_method]["test"].isna().any(axis=1)).sum()

# Regime edges come from the validation fit hours only
for split_method in SPLIT_METHODS:
    fit = MEMBER_TABLE[split_method]["validation"].loc[FIT_HOURS[split_method]]
    assert np.allclose(REGIME_EDGES[split_method][1:], member_mean(fit).quantile(LEVEL_QUANTILES).to_numpy())

print("section 3 self-check passed")
print(f"  methods {METHOD_KEYS}: fitted on validation hours only, weights >= 0 and summing to 1,")
print("  no forecast where a member is missing, adaptive covers the whole test year")

---

## <a id="sec-4"></a>4 Honest intervals

### 4.1 Out-of-fold validation forecasts

A method's in-sample validation errors are too small, because its parameters were fitted on the
same hours ([§3.7](#sec-3)). A band calibrated on them would be too narrow. Every method with
fitted parameters therefore also gets **out-of-fold** validation forecasts:

- the validation year is cut into **calendar-month blocks** (the partial first and last month are
  blocks of their own)
- for each block, every parameter is fitted on the **other** blocks and the block is forecast
- blocks rather than random hours, so neighbouring hours with similar errors cannot leak into the
  held-out part

What is refitted per block:

| method | refitted on the other months |
|---|---|
| `equal_mean` | nothing: it has no parameters, its validation forecast is already out-of-sample |
| `weighted` | the weights |
| `regime_weighted` | the regime edges, the global weights and every regime's weights |
| `adaptive` | the choice of the window; the weights themselves are out-of-sample by construction |
| `stacking` | the regime edges and the meta-model |

Fitting on "other months" uses later months of the validation year as well. That is fine here: it
never reaches a test hour, and it only estimates the size of the error. The **final** parameters
for the test year stay the ones of §3, fitted on the whole validation year.


In [ ]:
def levels_with(table, edges):
    """Regime of each hour against given edges (the rule of §3.1, with the edges as an argument)."""
    mean = member_mean(table)
    level = pd.Series(
        np.array(LEVEL_NAMES)[np.searchsorted(edges, mean.fillna(0).to_numpy(), side="right")], index=table.index
    )
    season = pd.Series(table.index.month.map(SEASON_OF_MONTH), index=table.index)
    if REGIMES["level"] and REGIMES["season"]:
        regime = level + " / " + season
    elif REGIMES["season"]:
        regime = season
    else:
        regime = level
    return regime.where(mean.notna())


def fit_regime_model(fit, actual):
    """Edges, global weights and shrunk regime weights from fit hours (§3.1, §3.3, §3.4)."""
    edges = np.r_[LEVEL_ZERO, member_mean(fit).quantile(LEVEL_QUANTILES).to_numpy()]
    global_weights = lp_weights(fit.to_numpy(), actual.to_numpy())
    regime = levels_with(fit, edges)
    weights = {}
    for name in REGIME_NAMES:
        in_regime = (regime == name).to_numpy()
        n = int(in_regime.sum())
        own = lp_weights(fit.to_numpy()[in_regime], actual.to_numpy()[in_regime]) if n else global_weights
        share = n / (n + SHRINK_HOURS) if n + SHRINK_HOURS > 0 else 1.0
        weights[name] = share * own + (1 - share) * global_weights
    return edges, global_weights, weights


def apply_regime_model(table, edges, weights):
    """Each hour combined with the weights of its regime; empty where a member is missing."""
    regime = levels_with(table, edges)
    rows = np.vstack([weights[name] if pd.notna(name) else np.full(len(MEMBER_KEYS), np.nan) for name in regime])
    return pd.Series((table[MEMBER_KEYS].to_numpy() * rows).sum(axis=1), index=table.index).where(
        table[MEMBER_KEYS].notna().all(axis=1)
    )


def stacking_with(table, edges):
    """Member forecasts plus one-hot regimes against given edges (§3.6)."""
    one_hot = pd.get_dummies(levels_with(table, edges)).reindex(columns=REGIME_NAMES, fill_value=False).astype(float)
    return pd.concat([table[MEMBER_KEYS], one_hot], axis=1)


# Month blocks of the validation year
VALIDATION_MONTHS = WINDOW_HOURS["validation"].to_period("M").unique()
OOF = {split_method: pd.DataFrame(index=WINDOW_HOURS["validation"]) for split_method in SPLIT_METHODS}
OOF_LOG = []  # one row per method, split method and held-out month: what it was fitted on

ADAPTIVE_RUNS = {}  # every candidate window's forecast, to choose the window per block
if "adaptive" in METHOD_KEYS:
    for split_method in SPLIT_METHODS:
        started = time.perf_counter()
        ADAPTIVE_RUNS[split_method] = {length: adaptive_run(split_method, length)[0] for length in ADAPTIVE_WINDOWS}
        add_cost("adaptive", split_method, calibration=time.perf_counter() - started)

for split_method in SPLIT_METHODS:
    validation = MEMBER_TABLE[split_method]["validation"]
    fit_hours = FIT_HOURS[split_method]
    month_of_fit = fit_hours.to_period("M")
    for key in METHOD_KEYS:
        if key == "equal_mean":
            OOF[split_method][key] = ENSEMBLE[split_method]["validation"]["equal_mean"]
            continue
        started = time.perf_counter()
        forecast = pd.Series(np.nan, index=validation.index)
        for month in VALIDATION_MONTHS:
            train = fit_hours[month_of_fit != month]
            held_out = validation.index[validation.index.to_period("M") == month]
            fit, actual = validation.loc[train], ACTUAL[train]
            if key == "weighted":
                weights = lp_weights(fit.to_numpy(), actual.to_numpy())
                forecast[held_out] = combine(validation.loc[held_out], weights)
            elif key == "regime_weighted":
                edges, _, weights = fit_regime_model(fit, actual)
                forecast[held_out] = apply_regime_model(validation.loc[held_out], edges, weights)
            elif key == "adaptive":
                runs = ADAPTIVE_RUNS[split_method]
                defined = pd.concat({l: run.reindex(train) for l, run in runs.items()}, axis=1).notna().all(axis=1)
                scored = train[defined.to_numpy()]
                mae = {l: (run[scored] - ACTUAL[scored]).abs().mean() for l, run in runs.items()}
                forecast[held_out] = runs[min(mae, key=mae.get)].reindex(held_out)
            elif key == "stacking":
                edges = np.r_[LEVEL_ZERO, member_mean(fit).quantile(LEVEL_QUANTILES).to_numpy()]
                model = Ridge(alpha=METHODS["stacking"]["alpha"]).fit(stacking_with(fit, edges), actual)
                block = validation.loc[held_out]
                complete = block.notna().all(axis=1)
                forecast[held_out[complete.to_numpy()]] = model.predict(stacking_with(block[complete], edges))
            OOF_LOG.append({"method": key, "split_method": split_method, "held_out": month,
                            "fit_months": sorted(set(train.to_period("M")))})
        OOF[split_method][key] = forecast
        add_cost(key, split_method, calibration=time.perf_counter() - started)

# In-sample against out-of-fold validation MAE
for split_method in SPLIT_METHODS:
    hours = COMMON["validation"]
    rows = {}
    for key in METHOD_KEYS:
        inside = ENSEMBLE[split_method]["validation"][key].reindex(hours)
        outside = OOF[split_method][key].reindex(hours)
        scored = hours[(inside.notna() & outside.notna()).to_numpy()]
        rows[METHODS[key]["label"]] = {
            "in-sample MAE": (inside[scored] - ACTUAL[scored]).abs().mean(),
            "out-of-fold MAE": (outside[scored] - ACTUAL[scored]).abs().mean(),
            "hours": len(scored),
        }
    table = pd.DataFrame(rows).T
    table["optimism (MWh)"] = table["out-of-fold MAE"] - table["in-sample MAE"]
    best = BEST_MEMBER[split_method]
    best_mae = (FORECASTS[split_method]["validation"].loc[hours, best] - ACTUAL[hours]).abs().mean()
    print(f"{split_method}: validation MAE, {len(VALIDATION_MONTHS)} month blocks; "
          f"best member {label_of(best)}: {best_mae:,.0f} MWh")
    display(table.style.format("{:,.0f}"))

### 4.2 The 95 % band

Spec 06 Behaviour 20's method, unchanged:

- **Residual sign**: the band uses `r = actual − forecast`, the opposite sign to the project's
  `error = forecast − actual`.
- **Quantiles per local hour of day**: the `(1 − level) / 2` and `(1 + level) / 2` quantiles of `r`
  over the calibration hours, since the error size varies strongly with the hour.
- **Band**: `forecast + [q_low, q_high]` for each test hour. A strongly biased hour gives both
  quantiles the same sign, so the band need not contain the forecast; that is correct.
- Calibrated once on the validation year, never updated during the test year: drift shows up as
  lost coverage in §5.

**Calibration hours per row:**

| row | calibrated on |
|---|---|
| each ensemble × split method | its **out-of-fold** validation residuals (§4.1) |
| each member × split method | its validation run of the same split method, as spec 06's `CALIBRATED_ON` |
| seasonal naive | its validation-year residuals |
| SMARD | none: a point forecast, no band |


In [ ]:
LOW_Q, HIGH_Q = (1 - INTERVAL["level"]) / 2, (1 + INTERVAL["level"]) / 2


def band_quantiles(forecast):
    """Per local hour: the LOW_Q and HIGH_Q quantiles of r = actual − forecast (spec 06 §6)."""
    r = (ACTUAL.reindex(forecast.index) - forecast).dropna()
    quantiles = r.groupby(r.index.hour).quantile([LOW_Q, HIGH_Q]).unstack()
    return quantiles.set_axis(["q_low", "q_high"], axis=1)


def with_band(forecast, quantiles):
    """The forecast plus its band, using the quantiles of each hour's local hour."""
    q = quantiles.reindex(forecast.index.hour).set_axis(forecast.index)
    return pd.DataFrame({"forecast": forecast, "lower": forecast + q["q_low"], "upper": forecast + q["q_high"]})


# (row key, split method) -> test-hour forecast with band, and the quantiles it used
BANDS, CALIBRATION = {}, {}
for split_method in SPLIT_METHODS:
    for key in METHOD_KEYS:
        row = (f"ensemble_{key}", split_method)
        quantiles = band_quantiles(OOF[split_method][key])
        CALIBRATION[row] = ("out-of-fold validation", quantiles)
        BANDS[row] = with_band(ENSEMBLE[split_method]["test"][key], quantiles)
    for key in SAVES:
        row = (key, split_method)
        quantiles = band_quantiles(FORECASTS[split_method]["validation"][key])
        CALIBRATION[row] = (f"validation_{split_method}", quantiles)
        BANDS[row] = with_band(FORECASTS[split_method]["test"][key], quantiles)
naive_quantiles = band_quantiles(FORECASTS[SPLIT_METHODS[0]]["validation"]["seasonal_naive"])
CALIBRATION[("seasonal_naive", "none")] = ("validation year", naive_quantiles)
BANDS[("seasonal_naive", "none")] = with_band(FORECASTS[SPLIT_METHODS[0]]["test"]["seasonal_naive"], naive_quantiles)

widths = pd.DataFrame(
    {
        (split_method, label): {
            "mean width (MWh)": (CALIBRATION[(key, split_method)][1]["q_high"] - CALIBRATION[(key, split_method)][1]["q_low"]).mean(),
            "narrowest hour (MWh)": (CALIBRATION[(key, split_method)][1]["q_high"] - CALIBRATION[(key, split_method)][1]["q_low"]).min(),
            "widest hour (MWh)": (CALIBRATION[(key, split_method)][1]["q_high"] - CALIBRATION[(key, split_method)][1]["q_low"]).max(),
            "calibrated on": CALIBRATION[(key, split_method)][0],
        }
        for split_method in SPLIT_METHODS
        for key, label in [(f"ensemble_{k}", METHODS[k]["label"]) for k in METHOD_KEYS]
        + [(BEST_MEMBER[split_method], f"{label_of(BEST_MEMBER[split_method])} (best member)")]
    }
).T
print(f"{INTERVAL['level']:.0%} band per row, from the per-hour quantiles (the width over the test year follows in §5)")
display(widths.style.format("{:,.0f}", subset=["mean width (MWh)", "narrowest hour (MWh)", "widest hour (MWh)"]))

The band per local hour for the split method in `PLOT_SPLIT_METHOD`: the `q_low` and `q_high`
quantiles of `r = actual − forecast`, for every enabled method and the best member. A narrower band
at the same coverage is the better band; whether the coverage holds is the question of §5.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))
rows = [(f"ensemble_{k}", METHODS[k]["label"], METHODS[k]["color"]) for k in METHOD_KEYS]
best = BEST_MEMBER[DIAG_SPLIT]
rows.append((best, f"{label_of(best)} (best member)", STYLE[best]["color"]))
for key, label, color in rows:
    quantiles = CALIBRATION[(key, DIAG_SPLIT)][1]
    linestyle = "--" if key == best else "-"
    ax.plot(quantiles.index, quantiles["q_high"], color=color, linestyle=linestyle, linewidth=1.6, label=label)
    ax.plot(quantiles.index, quantiles["q_low"], color=color, linestyle=linestyle, linewidth=1.6)
ax.axhline(0, color="0.4", linewidth=1)
ax.set_xticks(range(0, 24, 3))
ax.set_xlim(-0.5, 23.5)
ax.set_title(f"{INTERVAL['level']:.0%} band per local hour (r = actual − forecast) · {DIAG_SPLIT} split", fontsize=15, pad=12)
ax.set_xlabel("local hour", color="grey")
ax.set_ylabel("r = actual − forecast (MWh)", color="grey")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(colors="black", length=0)
ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.show()

### 4.3 Self-check


In [ ]:
# time_series is untouched
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# No out-of-fold forecast used a parameter fitted on its own month
log = pd.DataFrame(OOF_LOG)
assert all(row["held_out"] not in row["fit_months"] for _, row in log.iterrows())
for key in METHOD_KEYS:
    if key == "equal_mean":
        continue
    for split_method in SPLIT_METHODS:
        blocks = log[(log["method"] == key) & (log["split_method"] == split_method)]["held_out"]
        assert len(blocks) == len(VALIDATION_MONTHS), (key, split_method)

# Out-of-fold forecasts exist wherever the in-sample ones do; adaptive may lose more of its first
# days, if a block chose a longer window than the final one
for split_method in SPLIT_METHODS:
    for key in METHOD_KEYS:
        inside = ENSEMBLE[split_method]["validation"][key].notna()
        outside = OOF[split_method][key].notna()
        if key == "adaptive":
            print(f"  adaptive {split_method}: {int(outside.sum()):,} out-of-fold hours, {int(inside.sum()):,} in-sample")
        else:
            assert outside.equals(inside), (split_method, key)

# Every band is calibrated on validation hours only and applied to the test hours
for row, band in BANDS.items():
    assert band.index.equals(WINDOW_HOURS["test"]), row
    quantiles = CALIBRATION[row][1]
    assert set(quantiles.index) == set(range(24)), row
    has_forecast = band["forecast"].notna().to_numpy()
    width = (quantiles["q_high"] - quantiles["q_low"]).reindex(band.index.hour).to_numpy()
    assert np.allclose((band["upper"] - band["lower"]).to_numpy()[has_forecast], width[has_forecast]), row
    assert band.loc[~has_forecast, ["lower", "upper"]].isna().all().all(), row

# SMARD has no band
assert not any(key == "smard" for key, _ in BANDS)

print("section 4 self-check passed")
print(f"  {len(VALIDATION_MONTHS)} month blocks, no block fitted on itself; {len(BANDS)} banded rows, "
      "calibrated on validation hours only")

---

## <a id="sec-5"></a>5 Evaluation on the test year

The test year is scored **once**, with spec 06's definitions (Behaviours 21–25), on one common set
of hours for every row. Nothing in this section feeds back into a weight, edge, window or band.

### 5.1 Rows and common hours

Rows: every enabled ensemble × split method (`ensemble_<method>`), every member × split method,
SMARD and seasonal naive (`split_method = "none"`, the same in both split methods).

**Common hours**: test hours where the actual and **every** row have a forecast, a strict
intersection. SMARD is re-scored from `smard_forecast_errors_hourly.csv`.

Two references per row:

- **skill vs SMARD** `= 100 · (1 − MAE / MAE_SMARD)`, positive = better than SMARD
- **skill vs best member** `= 100 · (1 − MAE / MAE_best)`, against the split method's best member
  chosen on the **validation** year ([§2.5](#sec-2)); positive = better than picking that single
  model


In [ ]:
# (model, split_method) -> test forecast; ensembles are called ensemble_<method>
ROWS = {}
for split_method in SPLIT_METHODS:
    for key in METHOD_KEYS:
        ROWS[(f"ensemble_{key}", split_method)] = ENSEMBLE[split_method]["test"][key]
    for key in SAVES:
        ROWS[(key, split_method)] = FORECASTS[split_method]["test"][key]
ROWS[("smard", "none")] = SMARD_FORECAST.reindex(WINDOW_HOURS["test"])
ROWS[("seasonal_naive", "none")] = FORECASTS[SPLIT_METHODS[0]]["test"]["seasonal_naive"]

complete = ACTUAL.reindex(WINDOW_HOURS["test"]).notna() & pd.concat(ROWS, axis=1).notna().all(axis=1)
SCORED = WINDOW_HOURS["test"][complete.to_numpy()]
SCORE_ACTUAL = ACTUAL[SCORED]
SCORE_ERRORS = pd.DataFrame({row: forecast[SCORED] - SCORE_ACTUAL for row, forecast in ROWS.items()})
SCORE_ERRORS.columns = pd.MultiIndex.from_tuples(SCORE_ERRORS.columns, names=["model", "split_method"])

# SMARD's errors from its own file must agree with forecast − actual
assert np.allclose(SCORE_ERRORS[("smard", "none")], smard_errors.loc[SCORED, "err_residual_load"])


def row_label(row):
    """Display label of a (model, split_method) row."""
    model, split_method = row
    if model.startswith("ensemble_"):
        name = METHODS[model.removeprefix("ensemble_")]["label"]
        return f"Ensemble: {name} ({split_method})"
    return label_of(model) if split_method == "none" else f"{label_of(model)} ({split_method})"


def best_row(row):
    """The best-member row a row is compared with; None for the rows without a split method."""
    return None if row[1] == "none" else (BEST_MEMBER[row[1]], row[1])


lost = len(WINDOW_HOURS["test"]) - len(SCORED)
print(f"{len(ROWS)} rows scored on {len(SCORED):,} common test hours ({lost} test hours lost: a row without a forecast)")
print("best member per split method (chosen on validation): "
      + ", ".join(f"{s} {label_of(k)}" for s, k in BEST_MEMBER.items()))

### 5.2 Scoreboard A: accuracy

MAE, RMSE and bias over the common hours, the two skills, and the robustness view of spec 06: the
number of **full calendar months** inside the test window in which the row's MAE is below SMARD's,
and below the best member's. Sorted by MAE.

**Computational cost**, in seconds of wall-clock time, in the last four columns:

| column | single member (from its save's logs) | ensemble (measured in §3 and §4) |
|---|---|---|
| `tuning + calibration (s)` | the search on the validation walk-forward with the winner's validation run; for `static` also the `validation_static` run that calibrates its band | fitting the weights, edges, window or meta-model on the validation year, plus the out-of-fold refits for the band (§4) |
| `test run (s)` | every fit and forecast of the test year under the split method (1 fit static, one per `refit_every` rolling) | combining the members' test forecasts; `adaptive` chooses its window and forecasts in one step, so its test run is inside its tuning |
| `member models (s)` | – | the members' own `tuning + calibration` and `test run`, summed: an ensemble needs every member trained first. Shared by all ensembles of a split method |
| `total (s)` | the sum | the sum, members included |

The members were timed on the machine that **saved** them (`saved_at` and `device` in §1.5), the
ensembles on the machine running this notebook; read the two as orders of magnitude, not as a
benchmark. SMARD and seasonal naive have no cost rows.


In [ ]:
RECORDS = []  # model, split_method, table, metric, value, count: the scoreboard export (§6)


def add(row, table, metric, value, count):
    RECORDS.append({"model": row[0], "split_method": row[1], "table": table,
                    "metric": metric, "value": float(value) if pd.notna(value) else np.nan, "count": int(count)})


TEST_FULL_MONTHS = full_months("test")
monthly_mae = SCORE_ERRORS.abs().groupby(SCORED.to_period("M")).mean().loc[TEST_FULL_MONTHS]
smard_mae = SCORE_ERRORS[("smard", "none")].abs().mean()

for row in ROWS:
    error = SCORE_ERRORS[row]
    mae = error.abs().mean()
    add(row, "accuracy", "MAE", mae, len(SCORED))
    add(row, "accuracy", "RMSE", np.sqrt((error**2).mean()), len(SCORED))
    add(row, "accuracy", "bias", error.mean(), len(SCORED))
    if row != ("smard", "none"):
        add(row, "accuracy", "skill_pct", 100 * (1 - mae / smard_mae), len(SCORED))
        beats = int((monthly_mae[row] < monthly_mae[("smard", "none")]).sum())
        add(row, "accuracy", "months_beating_smard", beats, len(TEST_FULL_MONTHS))
    best = best_row(row)
    if best is not None:
        add(row, "accuracy", "skill_vs_best_pct", 100 * (1 - mae / SCORE_ERRORS[best].abs().mean()), len(SCORED))
        beats_best = int((monthly_mae[row] < monthly_mae[best]).sum())
        add(row, "accuracy", "months_beating_best", beats_best, len(TEST_FULL_MONTHS))


# Computational cost: the members' save logs, the ensembles' COST from §3 and §4
MEMBER_COST = {}
for key in REGISTRY_MEMBERS:
    result = joblib.load(MODEL_STORE / key / SAVE_FILES["results"])
    run_seconds = {
        run: float(log[["fit_seconds", "forecast_seconds"]].to_numpy().sum()) for run, (_, log) in result["runs"].items()
    }
    for split_method in SPLIT_METHODS:
        tuning = float(result["tuning"]["seconds"])  # the search plus the winner's validation_rolling run
        if split_method == "static":
            tuning += run_seconds["validation_static"]  # calibrates the static band
        MEMBER_COST[(key, split_method)] = {"tuning": tuning, "test": run_seconds[f"test_{split_method}"]}

COST_SECONDS = {}
for row in ROWS:
    model, split_method = row
    if model.startswith("ensemble_"):
        cost = COST[(model.removeprefix("ensemble_"), split_method)]
        members = sum(sum(MEMBER_COST[(key, split_method)].values()) for key in REGISTRY_MEMBERS)
        COST_SECONDS[row] = {"tuning": cost["fit"] + cost["calibration"], "test": cost["forecast"], "members": members}
    elif row in MEMBER_COST:
        COST_SECONDS[row] = {**MEMBER_COST[row], "members": 0.0}
for row, cost in COST_SECONDS.items():
    add(row, "accuracy", "tuning_seconds", cost["tuning"], 1)
    add(row, "accuracy", "test_run_seconds", cost["test"], 1)
    if row[0].startswith("ensemble_"):
        add(row, "accuracy", "member_seconds", cost["members"], len(REGISTRY_MEMBERS))
    add(row, "accuracy", "total_seconds", sum(cost.values()), 1)


def scoreboard_table(table, columns):
    """One table of the long records as rows × metrics, in Scoreboard A's MAE order."""
    records = pd.DataFrame(RECORDS)
    records = records[records["table"] == table]
    pivot = records.pivot_table(index=["model", "split_method"], columns="metric", values="value", sort=False)
    pivot = pivot.reindex(columns=list(columns)).reindex(ORDER)
    pivot.index = [row_label(row) for row in pivot.index]
    return pivot.rename(columns=columns)


accuracy = pd.DataFrame(RECORDS).query("table == 'accuracy' and metric == 'MAE'")
ORDER = list(accuracy.sort_values("value")[["model", "split_method"]].itertuples(index=False, name=None))

ACCURACY_COLUMNS = {
    "MAE": "MAE (MWh)", "RMSE": "RMSE (MWh)", "bias": "bias (MWh)",
    "skill_pct": "skill vs SMARD (%)", "skill_vs_best_pct": "skill vs best member (%)",
    "months_beating_smard": "months beating SMARD", "months_beating_best": "months beating best member",
    "tuning_seconds": "tuning + calibration (s)", "test_run_seconds": "test run (s)",
    "member_seconds": "member models (s)", "total_seconds": "total (s)",
}
table = scoreboard_table("accuracy", ACCURACY_COLUMNS)
print(f"Scoreboard A · {len(SCORED):,} common test hours · {len(TEST_FULL_MONTHS)} full months · sorted by MAE")
percent_columns = ["skill vs SMARD (%)", "skill vs best member (%)"]
second_columns = ["tuning + calibration (s)", "test run (s)", "member models (s)", "total (s)"]


def highlight_ensembles(row):
    return ["background-color: #EAF2FB" if row.name.startswith("Ensemble") else "" for _ in row]


display(
    table.style.apply(highlight_ensembles, axis=1)
    .format("{:,.0f}", na_rep="–", subset=[c for c in table.columns if c not in percent_columns + second_columns])
    .format("{:+.1f}", na_rep="–", subset=percent_columns)
    .format("{:,.1f}", na_rep="–", subset=second_columns)
)

### 5.3 Scoreboard B: extremes

Spec 06 Behaviour 24 on the same rows. Bin edges: the `{1, 25, 75, 99} %` quantiles of the
**actual** over all test hours, plus the fixed 0 MWh of `below_zero`, shared by every row.
MAE and bias per bin, **binned by the actual** and **binned by the row's own forecast**; the
forecast-binned table also shows each row's hours per bin. Skill is reported only where a row and
its reference are scored on the same hours: binned by the actual, and on the day extremes.

**The regression-to-the-mean rule applies**: only a tail difference that shows in both views
counts.

**Day maximum and minimum**: from the common hours only; a day counts if they cover at least
`DAY_COMPLETENESS` of its hours. The value error is `forecast extreme − actual extreme`.


In [ ]:
SCORE_EDGES = ACTUAL[WINDOW_HOURS["test"]].quantile(TAIL_LEVELS).to_numpy()  # all test hours, as in spec 06
p1, p25, p75, p99 = SCORE_EDGES
SCORE_BINS = ["low_extreme", "below_zero", "ordinary", "high_extreme"]


def score_masks(values):
    """The four bins with the test-window edges; below_zero overlaps low_extreme."""
    return {
        "low_extreme": values <= p1,
        "below_zero": values < LEVEL_ZERO,
        "ordinary": (values >= p25) & (values <= p75),
        "high_extreme": values > p99,
    }


by_actual = score_masks(SCORE_ACTUAL)
for row, forecast in ROWS.items():
    error = SCORE_ERRORS[row]
    by_forecast = score_masks(forecast[SCORED])
    best = best_row(row)
    for name in SCORE_BINS:
        actual_mask, forecast_mask = by_actual[name], by_forecast[name]
        mae = error[actual_mask].abs().mean()
        add(row, "extremes", f"MAE_{name}_by_actual", mae, actual_mask.sum())
        add(row, "extremes", f"bias_{name}_by_actual", error[actual_mask].mean(), actual_mask.sum())
        add(row, "extremes", f"MAE_{name}_by_forecast", error[forecast_mask].abs().mean(), forecast_mask.sum())
        add(row, "extremes", f"bias_{name}_by_forecast", error[forecast_mask].mean(), forecast_mask.sum())
        if row != ("smard", "none"):
            smard_bin = SCORE_ERRORS[("smard", "none")][actual_mask].abs().mean()
            add(row, "extremes", f"skill_pct_{name}_by_actual", 100 * (1 - mae / smard_bin), actual_mask.sum())
        if best is not None:
            best_bin = SCORE_ERRORS[best][actual_mask].abs().mean()
            add(row, "extremes", f"skill_vs_best_pct_{name}_by_actual", 100 * (1 - mae / best_bin), actual_mask.sum())

# Day extremes on the common hours
day_of = SCORED.normalize()
per_day = pd.Series(1, index=SCORED).groupby(day_of).size()
EXTREME_DAYS = per_day.index[per_day.ge(MIN_OBS_PER_DAY)]
in_days = day_of.isin(EXTREME_DAYS)
actual_by_day = SCORE_ACTUAL[in_days].groupby(day_of[in_days])
DAY_ERRORS = {}
for row, forecast in ROWS.items():
    forecast_by_day = forecast[SCORED][in_days].groupby(day_of[in_days])
    DAY_ERRORS[row] = {"max": forecast_by_day.max() - actual_by_day.max(), "min": forecast_by_day.min() - actual_by_day.min()}
for row in ROWS:
    best = best_row(row)
    for extreme in ["max", "min"]:
        error = DAY_ERRORS[row][extreme]
        add(row, "extremes", f"MAE_day_{extreme}", error.abs().mean(), len(error))
        add(row, "extremes", f"bias_day_{extreme}", error.mean(), len(error))
        if row != ("smard", "none"):
            smard_day = DAY_ERRORS[("smard", "none")][extreme].abs().mean()
            add(row, "extremes", f"skill_pct_day_{extreme}", 100 * (1 - error.abs().mean() / smard_day), len(error))
        if best is not None:
            best_day = DAY_ERRORS[best][extreme].abs().mean()
            add(row, "extremes", f"skill_vs_best_pct_day_{extreme}", 100 * (1 - error.abs().mean() / best_day), len(error))

counts = {name: int(by_actual[name].sum()) for name in SCORE_BINS}
print("edges (MWh): " + f"P1 {p1:,.0f} · P25 {p25:,.0f} · P75 {p75:,.0f} · P99 {p99:,.0f} · below_zero < {LEVEL_ZERO:g}")
print("hours by actual: " + ", ".join(f"{name} {count:,}" for name, count in counts.items()))

for view in ["actual", "forecast"]:
    columns = {}
    for name in SCORE_BINS:
        columns[f"MAE_{name}_by_{view}"] = (name, "MAE")
        columns[f"bias_{name}_by_{view}"] = (name, "bias")
        if view == "actual":
            columns[f"skill_pct_{name}_by_actual"] = (name, "skill vs SMARD %")
            columns[f"skill_vs_best_pct_{name}_by_actual"] = (name, "skill vs best %")
    table = scoreboard_table("extremes", columns)
    table.columns = pd.MultiIndex.from_tuples(table.columns)
    if view == "forecast":
        hours = pd.DataFrame(RECORDS).query("table == 'extremes'")
        for name in SCORE_BINS:
            subset = hours[hours["metric"] == f"MAE_{name}_by_forecast"].set_index(["model", "split_method"])["count"]
            table[(name, "hours")] = [subset[row] for row in ORDER]
        table = table.reindex(columns=pd.MultiIndex.from_product([SCORE_BINS, ["MAE", "bias", "hours"]]))
    print(f"Scoreboard B · binned by the {view}" + (" (hours: the row's own forecast in the bin)" if view == "forecast" else ""))
    percent = [c for c in table.columns if "%" in c[1]]
    display(
        table.style.apply(highlight_ensembles, axis=1)
        .format("{:,.0f}", na_rep="–", subset=[c for c in table.columns if c not in percent])
        .format("{:+.1f}", na_rep="–", subset=percent)
    )

columns = {}
for extreme in ["max", "min"]:
    for metric, label in [("MAE", "MAE"), ("bias", "bias"), ("skill_pct", "skill vs SMARD %"), ("skill_vs_best_pct", "skill vs best %")]:
        columns[f"{metric}_day_{extreme}"] = (f"day {extreme}", label)
table = scoreboard_table("extremes", columns)
table.columns = pd.MultiIndex.from_tuples(table.columns)
print(f"Scoreboard B · day extremes · {len(EXTREME_DAYS)} days with at least {MIN_OBS_PER_DAY} common hours")
percent = [c for c in table.columns if "%" in c[1]]
display(
    table.style.apply(highlight_ensembles, axis=1)
    .format("{:,.0f}", na_rep="–", subset=[c for c in table.columns if c not in percent])
    .format("{:+.1f}", na_rep="–", subset=percent)
)

### 5.4 Scoreboard C: intervals

Coverage: the share of common hours whose actual lies inside the row's band ([§4.2](#sec-4)),
against the nominal level; `coverage − nominal` in percentage points (negative = band too narrow);
and the mean band width over the common hours. SMARD is a point forecast and has no row.


In [ ]:
nominal = 100 * INTERVAL["level"]
for row in ROWS:
    if row not in BANDS:
        continue
    band = BANDS[row].loc[SCORED]
    inside = (SCORE_ACTUAL >= band["lower"]) & (SCORE_ACTUAL <= band["upper"])
    add(row, "intervals", "coverage_pct", 100 * inside.mean(), len(SCORED))
    add(row, "intervals", "mean_width", (band["upper"] - band["lower"]).mean(), len(SCORED))

table = scoreboard_table("intervals", {"coverage_pct": "coverage (%)", "mean_width": "mean width (MWh)"})
table = table.dropna(how="all")
table.insert(1, "coverage − nominal (pp)", table["coverage (%)"] - nominal)
print(f"Scoreboard C · nominal {nominal:.0f} % · {len(SCORED):,} common test hours")
display(
    table.style.apply(highlight_ensembles, axis=1)
    .format({"coverage (%)": "{:.1f}", "coverage − nominal (pp)": "{:+.1f}", "mean width (MWh)": "{:,.0f}"})
)

### 5.5 The weights

The final weights, fitted on the validation year (§3), per method and split method; for
`regime_weighted` one row per regime, for `adaptive` the mean over the test days and its range.
The last column names the most-weighted member of each regime, and whether it is also the
validation winner of that regime in §3.1.


In [ ]:
weights = pd.DataFrame(WEIGHT_RECORDS)
for split_method in SPLIT_METHODS:
    mine = weights[weights["split_method"] == split_method]
    rows = {}
    for method in ["equal_mean", "weighted"]:
        part = mine[mine["method"] == method]
        if len(part):
            rows[METHODS[method]["label"]] = part.set_index("member")["weight"]
    regime_winner, _ = regime_mae(split_method, "validation")
    winners = regime_winner[REGISTRY_MEMBERS + (["smard"] if "smard" in MEMBER_KEYS else [])].idxmin(axis=1)
    part = mine[mine["method"] == "regime_weighted"]
    for regime in REGIME_NAMES:
        values = part[part["regime"] == regime].set_index("member")["weight"]
        if len(values):
            rows[f"Regime: {regime}"] = values
    part = mine[mine["method"] == "adaptive"]
    if len(part):
        test_days = part[part["regime"].isin([f"{d:%Y-%m-%d}" for d in TEST_DAYS])]
        rows["Adaptive: test-day mean"] = test_days.groupby("member")["weight"].mean()
        rows["Adaptive: test-day min"] = test_days.groupby("member")["weight"].min()
        rows["Adaptive: test-day max"] = test_days.groupby("member")["weight"].max()
    table = pd.DataFrame(rows).T.reindex(columns=MEMBER_KEYS)
    top = table.idxmax(axis=1)
    table.columns = [label_of(key) for key in table.columns]
    table["most weight"] = [
        "" if name.endswith(("min", "max")) else label_of(key) for name, key in top.items()
    ]
    table["= validation winner"] = [
        ("yes" if top[name] == winners.get(name.removeprefix("Regime: ")) else "no") if name.startswith("Regime: ") else ""
        for name in table.index
    ]
    member_columns = [label_of(key) for key in MEMBER_KEYS]
    print(f"{split_method}: final weights")
    display(table.style.format("{:.2f}", subset=member_columns)
            .background_gradient(cmap="Greens", subset=member_columns, vmin=0, vmax=1))

### 5.6 Forecast plot with the band

Two weeks of the test window, for `PLOT_METHOD` × `PLOT_SPLIT_METHOD` (`None`: the enabled ensemble
with the lowest test MAE): the local Monday–Sunday week containing the **highest** actual hour and
the one containing the **lowest**, clipped at the window's edges. Each panel shows the actual, SMARD,
the ensemble with its 95 % band and the best member.


In [ ]:
ensembles = [row for row in ORDER if row[0].startswith("ensemble_") and row[1] == DIAG_SPLIT]
PLOT_ROW = (f"ensemble_{PLOT_METHOD}", DIAG_SPLIT) if PLOT_METHOD else ensembles[0]
assert PLOT_ROW in BANDS, f"PLOT_METHOD {PLOT_METHOD!r} is not an enabled method"
best = best_row(PLOT_ROW)

hours = WINDOW_HOURS["test"]
week_rule = {"highest": ACTUAL[hours].idxmax(), "lowest": ACTUAL[hours].idxmin()}
fig, axes = plt.subplots(2, 1, figsize=(15, 10))
for ax, (name, stamp) in zip(axes, week_rule.items()):
    monday = stamp.normalize() - pd.Timedelta(days=stamp.dayofweek)
    week = hours[(hours >= monday) & (hours < monday + pd.Timedelta(days=7))]
    band = BANDS[PLOT_ROW].loc[week]
    method = METHODS[PLOT_ROW[0].removeprefix("ensemble_")]
    ax.fill_between(week, band["lower"], band["upper"], color=method["color"], alpha=0.15,
                    label=f"{INTERVAL['level']:.0%} band")
    ax.plot(week, ACTUAL[week], color=STYLE["actual"]["color"], linewidth=2, label=STYLE["actual"]["label"])
    ax.plot(week, SMARD_FORECAST[week], linewidth=1.4, **line_style("smard"))
    ax.plot(week, ROWS[best][week], linewidth=1.3, alpha=0.9,
            **(line_style(best[0]) | {"label": f"{label_of(best[0])} (best member)", "linestyle": ":"}))
    ax.plot(week, band["forecast"], color=method["color"], linewidth=1.8, label=row_label(PLOT_ROW))
    ax.axhline(0, color="0.6", linewidth=0.8)
    style_timeseries(ax, f"Week of the {name} actual hour ({stamp:%Y-%m-%d %H:%M}, {ACTUAL[stamp]:,.0f} MWh)", "residual load (MWh)")
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
    ax.xaxis.set_minor_locator(mdates.HourLocator((6, 12, 18)))
axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
fig.tight_layout()
print(f"plotted: {row_label(PLOT_ROW)}; weeks: Monday–Sunday around the highest and lowest actual test hour")
plt.show()

### 5.7 Self-check


In [ ]:
# time_series is untouched
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

records = pd.DataFrame(RECORDS)
value = records.set_index(["model", "split_method", "table", "metric"])["value"]

# Scoreboard MAE and RMSE match a direct recomputation from the forecasts
for row, forecast in ROWS.items():
    error = forecast[SCORED] - ACTUAL[SCORED]
    assert np.isclose(value[(*row, "accuracy", "MAE")], error.abs().mean()), row
    assert np.isclose(value[(*row, "accuracy", "RMSE")], np.sqrt((error**2).mean())), row

# SMARD's MAE matches its own file; a best member scores 0 % against itself
assert np.isclose(value[("smard", "none", "accuracy", "MAE")], smard_errors.loc[SCORED, "err_residual_load"].abs().mean())
for split_method, key in BEST_MEMBER.items():
    assert np.isclose(value[(key, split_method, "accuracy", "skill_vs_best_pct")], 0)

# Every row is scored on the same hours; nothing outside the test year
assert records.query("table == 'accuracy' and metric == 'MAE'")["count"].eq(len(SCORED)).all()
assert SCORED.isin(WINDOW_HOURS["test"]).all()

# Every ensemble and member row has a measured cost; the ensembles' totals include their members
for row in ROWS:
    if row[1] != "none":
        assert row in COST_SECONDS, row
        assert value[(*row, "accuracy", "total_seconds")] >= 0, row
for row, cost in COST_SECONDS.items():
    if row[0].startswith("ensemble_"):
        expected = sum(sum(MEMBER_COST[(k, row[1])].values()) for k in REGISTRY_MEMBERS)
        assert np.isclose(cost["members"], expected), row

# Coverage lies in [0, 100]; SMARD has no interval row
intervals = records[records["table"] == "intervals"]
assert intervals.query("metric == 'coverage_pct'")["value"].between(0, 100).all()
assert not ((intervals["model"] == "smard")).any()

print("section 5 self-check passed")
print(f"  {len(ROWS)} rows on {len(SCORED):,} common hours; MAE / RMSE recomputed; SMARD matches its file;")
print("  best members score 0 % against themselves; coverage within [0, 100]")

---

## <a id="sec-6"></a>6 Export

Written only when `EXPORT_ENABLED` is on (the default, as in spec 06); the frames are always built,
so the closing self-check works either way. All three files go to `data/models/`, plain CSV
(`sep=","`, `decimal="."`, UTF-8), gitignored by the `data/models/*.csv` rule. Nothing is written to
the `model_*` files or to a save.

| file | content |
|---|---|
| `ensemble_forecast_errors_hourly.csv` | one row per test hour × ensemble × split method: `timestamp`, `model` (`ensemble_<method>`), `split_method`, `residual_load`, `forecast`, `lower`, `upper`, `err_residual_load`; spec 06's hourly layout. Members and SMARD stay in their own files. |
| `ensemble_scoreboard.csv` | spec 06's long scoreboard layout (`model`, `split_method`, `table`, `metric`, `value`, `count`) for every row of §5, on this notebook's common hours, including the added `skill_vs_best_pct` and `months_beating_best` |
| `ensemble_weights.csv` | `method`, `split_method`, `regime`, `member`, `weight`; `adaptive` has one regime per delivery day |


In [ ]:
EXPORT_PATHS = {
    "hourly": MODEL_STORE / "ensemble_forecast_errors_hourly.csv",
    "scoreboard": MODEL_STORE / "ensemble_scoreboard.csv",
    "weights": MODEL_STORE / "ensemble_weights.csv",
}
STAMP = "%Y-%m-%d %H:%M:%S"

hourly_parts = []
for (model, split_method), band in BANDS.items():
    if not model.startswith("ensemble_"):
        continue
    part = band.copy()
    part.insert(0, "residual_load", ACTUAL.reindex(part.index))
    part["err_residual_load"] = part["forecast"] - part["residual_load"]
    part.insert(0, "split_method", split_method)
    part.insert(0, "model", model)
    hourly_parts.append(part.rename_axis("timestamp").reset_index())
EXPORT = {
    "hourly": pd.concat(hourly_parts, ignore_index=True)[
        ["timestamp", "model", "split_method", "residual_load", "forecast", "lower", "upper", "err_residual_load"]
    ],
    "scoreboard": pd.DataFrame(RECORDS)[["model", "split_method", "table", "metric", "value", "count"]],
    "weights": pd.DataFrame(WEIGHT_RECORDS)[["method", "split_method", "regime", "member", "weight"]],
}

if EXPORT_ENABLED:
    for name, frame in EXPORT.items():
        frame.to_csv(EXPORT_PATHS[name], index=False, date_format=STAMP)
        print(f"wrote {EXPORT_PATHS[name].relative_to(DATA_DIR.parent).as_posix()}: {len(frame):,} rows")
else:
    print("EXPORT_ENABLED is off: nothing written. With it on, the notebook writes")
    for path in EXPORT_PATHS.values():
        print(f"  {path.relative_to(DATA_DIR.parent).as_posix()}")

---

## <a id="sec-7"></a>7 Closing checks and conclusion

### 7.1 Closing self-check

In addition to the self-checks at the end of every section (spec 08 Behaviour 23):

- no parameter, edge, window or band was fitted on a test hour
- weights are `≥ 0` and sum to 1 for every method, regime and adaptive day
- `equal_mean` is the row mean of the members; every ensemble is empty exactly where a member is
- **truncation test for `adaptive`**: for `LEAKAGE_SAMPLE_DAYS` test days drawn with `SEED`, the
  day's weights are recomputed from the actuals **with every hour after that day's cutoff masked**;
  they must be identical. A weight that read a later actual would change.
- the member forecasts in memory are still the saves they came from, and seasonal naive follows
  spec 06's rule
- scoreboard MAE and RMSE match a direct recomputation from the hourly export frame, and SMARD's MAE
  matches `smard_forecast_errors_hourly.csv`
- with `EXPORT_ENABLED` on, the written files read back with a bare `pd.read_csv` equal the frames
- `time_series` still carries exactly `SERIES + DERIVED`, with the row count and span of loading


In [ ]:
LEAKAGE_SAMPLE_DAYS = 20

# 1. Nothing fitted on a test hour
for split_method in SPLIT_METHODS:
    assert not FIT_HOURS[split_method].isin(WINDOW_HOURS["test"]).any()
for row, (source, quantiles) in CALIBRATION.items():
    assert "test" not in source, row

# 2. Weights >= 0, summing to 1
weights = pd.DataFrame(WEIGHT_RECORDS)
sums = weights.groupby(["method", "split_method", "regime"])["weight"].sum()
assert np.allclose(sums, 1, atol=1e-9) and (weights["weight"] >= -1e-12).all()

# 3. equal_mean is the row mean; empty exactly where a member is empty
for split_method in SPLIT_METHODS:
    for window in WINDOW_ORDER:
        members = MEMBER_TABLE[split_method][window]
        missing = members.isna().any(axis=1)
        for key in METHOD_KEYS:
            assert ENSEMBLE[split_method][window][key][missing].isna().all(), (split_method, window, key)
        if "equal_mean" in METHOD_KEYS:
            assert np.allclose(ENSEMBLE[split_method][window]["equal_mean"][~missing], members.mean(axis=1)[~missing])


# 4. Truncation test for adaptive: weights from actuals masked after the day's cutoff
def adaptive_weights_from(split_method, length, day, actual):
    """The adaptive weights of one day, recomputed directly from a given actual series."""
    table = pd.concat([MEMBER_TABLE[split_method][window] for window in WINDOW_ORDER])
    cutoff = FORECAST_SETTING.at[day, "cutoff"]
    window = table.index[(table.index >= cutoff - length) & available(table.index, cutoff)]
    mae = table.loc[window].sub(actual.reindex(window), axis=0).abs().mean()
    return (1 / mae) / (1 / mae).sum()


rng = np.random.default_rng(SEED)
sample = pd.DatetimeIndex(sorted(rng.choice(TEST_DAYS, size=min(LEAKAGE_SAMPLE_DAYS, len(TEST_DAYS)), replace=False)))
truncation = []
for split_method, entry in ADAPTIVE.items():
    for day in sample:
        cutoff = FORECAST_SETTING.at[day, "cutoff"]
        masked = ACTUAL.where(available(ACTUAL.index, cutoff))  # every actual after the cutoff removed
        recomputed = adaptive_weights_from(split_method, entry["window"], day, masked)
        stored = entry["weights"].loc[day]
        truncation.append({"split_method": split_method, "day": day,
                           "max_abs_difference": float(np.abs(recomputed.to_numpy() - stored.to_numpy()).max())})
truncation = pd.DataFrame(truncation)
if len(truncation):
    assert (truncation["max_abs_difference"] < 1e-12).all(), truncation.sort_values("max_abs_difference").tail()

# 5. Member forecasts are still the saves; seasonal naive follows spec 06's rule
for split_method, windows in FORECASTS.items():
    for window, table in windows.items():
        for key in SAVES:
            assert table[key].equals(SAVES[key]["runs"][f"{window}_{split_method}"])
        lagged = ACTUAL.reindex(table.index - NAIVE_LAG).to_numpy()
        assert np.allclose(table["seasonal_naive"].to_numpy(), lagged, equal_nan=True)

# 6. Scoreboard against the hourly export frame; SMARD against its file
hourly = EXPORT["hourly"].set_index(["model", "split_method", "timestamp"])
value = pd.DataFrame(RECORDS).set_index(["model", "split_method", "table", "metric"])["value"]
assert np.allclose(hourly["err_residual_load"], hourly["forecast"] - hourly["residual_load"], equal_nan=True)
for (model, split_method), part in hourly.groupby(level=["model", "split_method"]):
    error = part.droplevel([0, 1]).loc[SCORED, "err_residual_load"]
    assert np.isclose(value[(model, split_method, "accuracy", "MAE")], error.abs().mean())
    assert np.isclose(value[(model, split_method, "accuracy", "RMSE")], np.sqrt((error**2).mean()))
assert np.isclose(value[("smard", "none", "accuracy", "MAE")], smard_errors.loc[SCORED, "err_residual_load"].abs().mean())

# 7. Round trip of the written files
if EXPORT_ENABLED:
    for name, frame in EXPORT.items():
        back = pd.read_csv(EXPORT_PATHS[name])
        if "timestamp" in back:
            back["timestamp"] = pd.to_datetime(back["timestamp"], format=STAMP)
        assert len(back) == len(frame), name
        assert list(back.columns) == list(frame.columns), name
        numeric = frame.select_dtypes("number").columns
        assert np.allclose(back[numeric].to_numpy(dtype=float), frame[numeric].to_numpy(dtype=float), equal_nan=True), name

# 8. time_series is untouched
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert (len(time_series), time_series.index.min(), time_series.index.max()) == (LOADED["rows"], LOADED["start"], LOADED["end"])

print("closing self-check passed")
print(f"  nothing fitted on a test hour; weights >= 0 and summing to 1; ensembles empty only where a member is")
if len(truncation):
    print(f"  adaptive truncation test: {len(sample)} test days per split method, max difference "
          f"{truncation['max_abs_difference'].max():.1e} (weights see no actual after the cutoff)")
print(f"  members equal their saves; scoreboard recomputed from the hourly export; "
      f"{'files read back equal the frames' if EXPORT_ENABLED else 'export off, no files checked'}")

### <a id="sec-7-2"></a>7.2 Does combining help?

> Written for the **default configuration** (seven members incl. SMARD, `SHRINK_HOURS = 100`,
> adaptive windows 7 / 30 days) and the saves of 2026-09-30: validation 2024-09-07 .. 2025-09-06,
> test 2025-09-07 .. 2026-09-06, 8,758 common test hours. The tables above always show the current
> run; after a change in §1.3, check the numbers here against them.

- **Yes: every ensemble beats every single member on the test year** (§5.2).
  - Best row: `regime_weighted`, rolling, MAE 2,277 MWh against the best member chosen on validation
    (`lgbm_direct`, 2,363 MWh): **+3.7 %**, ahead in 10 of 11 full months; +18.4 % against SMARD
    (2,790 MWh).
  - Static: `regime_weighted` 2,312 MWh, +7.7 % against `xgb_direct` (2,505 MWh), ahead in all 11
    months.
  - Even the member that happens to be best on test, `random_forest_hybrid` rolling (2,333 MWh),
    is behind all four rolling ensembles (2,277–2,292 MWh). Picking it would have needed the test
    year (§2.5).

- **The plain mean already does most of the work.**
  - `equal_mean` rolling: 2,289 MWh (+3.1 % against the best member). `weighted` is not better
    (2,292 MWh), as §4.1 foresaw out of fold (2,090 against 2,089 MWh).
  - `regime_weighted` adds 12 MWh over the mean (rolling) and 16 MWh (static); out of fold on the
    validation year the gain was 11 and 19 MWh, so the small edge held.
  - `adaptive` with `1 / MAE` stays close to equal weights (13–15 % per member on average, §5.5)
    and lands between the two (2,285 MWh).

- **The tails are mixed: strong below 0 MWh and at the top, weak in the deepest lows** (§5.3, rolling).
  - `below_zero` (563 h): 2,853 MWh against SMARD's 4,127 MWh (+30.9 %); it holds binned by the
    forecast too (2,832 MWh on 539 h against 3,816 MWh on 485 h).
  - `high_extreme` (88 h): 1,638 MWh against SMARD's 2,305 MWh (+29.0 %), by forecast 1,540 against
    1,650 MWh; the single `xgb_hybrid` is better still (1,540 MWh by actual).
  - `low_extreme` (88 h, ≤ −7,844 MWh): 3,900 MWh, barely ahead of SMARD (3,952 MWh, +1.3 %) and
    far behind `random_forest_hybrid` (2,973 MWh); `equal_mean` does better (3,559 MWh). The
    ensemble forecasts that bin too rarely (55 hours by forecast against 88 by actual) and stays
    too high there (bias +3,857 MWh): averaging pulls the deepest lows up.
  - Day extremes: day minimum 2,425 against SMARD's 3,364 MWh (+27.9 %), day maximum 1,961 against
    2,212 MWh (+11.3 %).

- **The "who wins where" pattern is only partly stable** (§2.4, §3.1).
  - Spec 09's pick rule: rank 1 is the same in validation and test in 3 of 5 categories under
    `rolling`, in 1 of 5 under `static`.
  - By the regimes of `regime_weighted`: the best member is the same in 1 of 5 regimes under
    `rolling`, 3 of 5 under `static`. The rolling `below_zero` weights rest on 140 validation hours
    and favour `lgbm_hybrid` (0.36) and `xgb_direct` (0.27), while on test `random_forest_hybrid` and
    `linear_direct` lead the deepest hours: consistent with the `low_extreme` weakness above.

- **The bands are a little too narrow, like the members' bands** (§5.4).
  - Coverage 92.0–92.5 % against the nominal 95 % for every ensemble; the members range from 89.9
    to 93.0 %. The gap comes from drift between validation and test year, not from the combination.
  - At the same coverage the ensemble bands are narrower: 10,502–10,600 MWh mean width against
    10,699–11,062 MWh for the members.

- **Read the result with its framing.**
  - SMARD is a member (weight 0.10 in the rolling `weighted` ensemble), and every member uses
    SMARD's component forecasts: the ensemble is a **post-processor of SMARD's forecast**, not an
    independent forecast, and it cannot run without SMARD.
  - It rests on **one test year**; the monthly counts (10 or 11 of 11 against the best member) are
    the robustness view. Tail statements rest on 88 hours per bin and stay qualitative.
  - A full run takes about 4 minutes, most of it the out-of-fold refits of §4.

- **Combining is cheap; the members are the cost** (§5.2, `total (s)`).
  - The rolling members took 1,699 s to tune and run, the static ones 1,465 s (timed on the
    machine that saved them); `random_forest_hybrid` alone accounts for 934 s (rolling).
  - On top of that, the combination adds 0 s (`equal_mean`) to 79 s (`regime_weighted`, rolling,
    almost all of it the out-of-fold refits for the band): at most about 5 % of the members' time.
  - A single member is far cheaper than an ensemble: `linear_direct` needs 7 s, `lgbm_direct`
    220 s. The +3.7 % of the best ensemble therefore costs roughly eight times the run time of the
    best single member (1,778 against 220 s, rolling).

**To try next (untested):**

- `SHRINK_HOURS` higher (e.g. 300–500): the regime pattern was unstable, so stronger shrinkage
  toward the global weights may keep the gain and lose less in `low_extreme`.
- `USE_SMARD_MEMBER = False`: SMARD's weight is small; whether the ensemble needs it is open (spec 08,
  Open questions).
- `MEMBERS` without one of the near-identical hybrids (error correlation 0.98, §2.3).
- For the low risk flag, `equal_mean` or `random_forest_hybrid` alone may be the better input than
  `regime_weighted`; the risk flags of §2.7 on the ensemble itself are not part of this notebook.
